In [ ]:
# Uninstall the ccogitarelink package
#! pip uninstall cogitarelink -y

In [ ]:
## Install fresh from the GitHub Package
#! pip install --no-cache-dir -U git+https://github.com/la3d/cogitarelink.git

In [ ]:
#| default_exp memory
#| export
import json
from pyld import jsonld
import httpx
from rdflib import Graph, Dataset, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from datetime import datetime
from enum import Enum
from typing import List, Literal, Optional, Dict, Any
from pydantic import BaseModel, Field
from fastcore.basics import patch
from dataclasses import dataclass, field
from claudette import Chat, toolloop, models
import time
import copy
from pathlib import Path
import types
import pickle

In [ ]:
from cogitarelink.memory import *
from cogitarelink.retriever import *
from cogitarelink.memorytools import *
from cogitarelink.vocabtools import *
from cogitarelink.utils import *
from cogitarelink.vocabtools import load_context_for_vocabulary, compact_entity_with_vocabulary, VOCABULARY_REGISTRY

## Load Cogitarelink source code

In [ ]:
memory_py = load_module_source("memory")

Loaded memory from /app/data/.local/lib/python3.11/site-packages/cogitarelink/memory.py (97668 characters)


In [ ]:
retriever_py = load_module_source("retriever")

Loaded retriever from /app/data/.local/lib/python3.11/site-packages/cogitarelink/retriever.py (53668 characters)


In [ ]:
vocabtools_py = load_module_source("vocabtools")

Loaded vocabtools from /app/data/.local/lib/python3.11/site-packages/cogitarelink/vocabtools.py (28798 characters)


**AI Prompt**

Hi solveit! I am Chuck, and I am working with you today! I'm an experienced Python coder, experienced with semantic web technologies like OWL,RDF and SPARQL, and I enjoy learning new things. When writing code for me, please follow fast.ai coding style: avoid comments; place single line if/loop/etc constructs on the same line; use concise variable names where appropriate. Follow the style you will see in my messages, as we work on this dialogue together. I am the author of the cogitarelink package and that allows LLM agents use JSON-LD 1.1 as a semantic memory system and to retrieve Linked Open Data and organize that LOD in LLM context windows.

**AI Response**

Hi Chuck! Great to meet you. I'm excited to work with you on cogitarelink. I see you've loaded the core modules from your package, which uses JSON-LD 1.1 for semantic memory and LOD retrieval.

What aspect of cogitarelink would you like to explore or improve today? I notice you're using Pydantic models, RDFlib, and pyld for working with JSON-LD. I'm curious about how you're structuring the memory system or if there's a specific feature you'd like to focus on.

Would you like to walk through a particular component of the system, test a feature, or perhaps develop something new?

Based on the source code provided, I can see that cogitarelink is a package designed to work with semantic data using JSON-LD 1.1 and Linked Open Data (LOD). Let me break down the main modules:

## memory.py

This module implements a semantic memory system that stores and retrieves JSON-LD data.

**Purpose:**
- Creates a unified memory store for JSON-LD data with context awareness
- Handles different types of resources (vocabularies, contexts, data sources, named graphs)
- Provides efficient indexing and querying mechanisms
- Manages context relationships, including scoped contexts

**Key features:**
- Resource caching with TTL (time-to-live) settings
- Context registry that tracks relationships between contexts
- Multiple indexing strategies (by ID, type, label, description)
- Named graph support for context isolation
- Follow-your-nose linked data traversal

**Coding style:**
- Uses `@patch` decorator from fastcore for extending classes
- Concise variable names
- Limited comments, focusing on function docstrings
- Single-line conditionals where appropriate
- Heavy use of Python's type system and dataclasses

## vocabtools.py

This module provides tools for working with different vocabularies in JSON-LD.

**Purpose:**
- Handles different vocabularies that may not follow standard linked data principles
- Resolves context collisions between different vocabularies
- Provides helpers for creating and working with datasets using specific vocabularies

**Key features:**
- Vocabulary registry with metadata about different vocabularies
- Custom document loader for PyLD that handles vocabularies differently based on their support level
- Collision strategies for combining vocabularies
- Helper functions for working with datasets using specific vocabularies

**Coding style:**
- Similar to memory.py - concise, function-focused
- Good use of type hints
- Minimal comments with descriptive function names
- Clear organization of related functions

## Relationship between modules:

The `memory.py` module provides the core semantic memory functionality, while `vocabtools.py` extends it with vocabulary-aware tools. The `SemanticMemory` class in memory.py uses functions from vocabtools.py (like `register_vocab_aware_loader` and `compact_entity_with_vocabulary`) to handle vocabulary-specific operations.

The code is written in a functional style with clear separation of concerns. It follows a "composition over inheritance" approach, using functions that operate on the `SemanticMemory` class rather than extending it through inheritance.

Would you like me to dive deeper into any specific aspect of these modules?

**AI Prompt**

Here is the source code to the $`vocabtools_py`. Could you familiarize yourself with the source code and tell me about the purpose of this module, and how it was written (coding style) etc?

**AI Response**

Looking at the `vocabtools_py` module, I can see it's designed to solve vocabulary management challenges in JSON-LD processing. Here's my analysis:

## Purpose of vocabtools.py

This module addresses a critical problem in the semantic web ecosystem - working with multiple vocabularies that may have different implementation patterns or support levels. It provides:

1. A vocabulary registry system that tracks metadata about different vocabularies
2. Custom document loaders for PyLD that handle vocabularies based on their support level
3. Collision resolution strategies when multiple vocabularies are used together
4. Helper functions for creating, storing, and retrieving data with specific vocabularies

The module particularly shines when dealing with vocabularies that:
- Don't follow standard linked data principles
- Have non-standard context locations
- Use JSON-LD 1.1 features like scoped contexts
- Require special handling for dereferencing

## Coding Style

The code follows the fast.ai style with these characteristics:

- Concise, domain-specific variable names (embeds, vocab_name)
- Single-line conditionals and short function definitions where appropriate
- Function-focused approach with clear docstrings explaining purpose
- No inline comments - code is self-documenting with descriptive function names
- Comprehensive type hints for all functions
- Functional programming patterns with pure functions where possible
- Dictionary definitions that prioritize readability

The module is organized in logical sections:
1. Registry definitions (VOCABULARY_REGISTRY, COLLISION_STRATEGIES)
2. Document loader implementation
3. Collision handling functions
4. Helper functions for working with vocabularies

## Key Components

The VOCABULARY_REGISTRY is particularly interesting - it stores metadata about vocabularies like:
- URI patterns
- Support levels ("direct", "cache", "discover")
- Context locations
- Access patterns
- Whether the vocabulary uses protected terms

The custom document loader provides fallback mechanisms when vocabularies can't be dereferenced directly, with three support levels:
1. Direct support - vocabularies with known context locations
2. Cache support - vocabularies that can be dereferenced but benefit from caching
3. Discovery support - unknown vocabularies where structure must be discovered

The collision strategies handle cases where vocabularies might have conflicting terms, using techniques like property scoping and graph partitioning.

Would you like me to explore any specific aspect of this module in more detail?

In [ ]:
import dspy # Add dspy import

# Configure DSPy LM for this notebook's context
try:
    # Use the same model as in 01_retriever.ipynb
    lm = dspy.LM('anthropic/claude-3-5-sonnet-20241022')
    dspy.configure(lm=lm)
    print("DSPy configured successfully.")
except Exception as e:
    print(f"Error configuring DSPy: {e}")
    # Optionally raise the error or handle it if configuration is critical
    # raise e

DSPy configured successfully.


## Semantic Layers in Our System
Foundational Vocabularies (Bootstrap Layer)

RDF, RDFS, OWL, SKOS
These provide the basic semantic framework (types, labels, subclasses)
LLMs need to understand these first to make sense of anything else
Mid-Level Vocabularies

Schema.org, PROV-O, Dublin Core
Build on the foundational layer with common concepts
Widely used across domains
Domain-Specific Vocabularies

VC, EPCIS, domain ontologies
Rely on both previous layers for meaning
The current implementation in memory_py does handle some of this through RDFlib by:

Using RDFS.label for label indexing
Using RDF.type for type indexing
Tracking subclass relationships with RDFS.subClassOf
However, when we partition the graph, we need to ensure:

We maintain these critical semantic relationships across partitions
We preserve the original identifiers that might be meaningful
We enable cross-partition querying that understands these semantic relationships
The issue isn't just about preserving IDs - it's about maintaining the semantic web of meaning that allows an LLM to understand the data holistically, even when it's partitioned.

Would you like me to focus on enhancing the querying mechanism to better handle these semantic relationships across partitions?

You've identified the key challenge here - balancing proper JSON-LD handling with the needs of LLMs to understand the content. Let me explain what an ideal graph partitioning approach should do:

## What Graph Partitioning Should Do
Graph partitioning in this context means:

Breaking a complex JSON-LD document with multiple contexts into separate entities
Each entity only uses one context, avoiding collision issues
Preserving the relationships between entities using ID references
Maintaining the semantic meaning of the original document
The benefit is that this creates a structure that both follows JSON-LD 1.1 best practices and can be processed by PyLD without context collisions.

Making It Flexible
For maximum flexibility, the partitioning should:

Be context-aware rather than pattern-specific (not just hardcoded for VC)
Analyze the contexts to determine compatibility
Intelligently assign properties to the most appropriate entities
Support multiple levels of nesting with consistent reference patterns
Allow for reconstruction of the original logical structure when needed
Would you like me to continue with how to query across partitions and handle vocabulary loading for LLMs?

Querying Across Graph Partitions
To effectively query across graph partitions:

Graph-Aware Querying: The query system needs to be aware of the graph structure and follow ID references between entities.

Reference Resolution: When retrieving an entity, automatically follow references to related entities (similar to database joins).

Context Recreation: For presenting to LLMs, reconstruct the logical structure of related entities while preserving their original contexts.

Named Graph Support: Use RDF named graphs to maintain separation while allowing cross-graph queries through the dataset.

Vocabulary Loading for LLMs
For LLMs to understand partitioned data:

Vocabulary Identification: When preparing data for an LLM, identify all vocabularies/contexts used across the partitions.

Context Preprocessing: Preprocess contexts to simplify and make them more LLM-friendly (remove excessive URIs, focus on term meanings).

Relationship Explanation: Include metadata that explains the relationships between partitioned entities.

Namespace Disambiguation: Provide clear markers for which terms come from which vocabulary to avoid confusion.
Querying Across Graph Partitions
To effectively query across graph partitions:

Graph-Aware Querying: The query system needs to be aware of the graph structure and follow ID references between entities.

Reference Resolution: When retrieving an entity, automatically follow references to related entities (similar to database joins).

Context Recreation: For presenting to LLMs, reconstruct the logical structure of related entities while preserving their original contexts.

Named Graph Support: Use RDF named graphs to maintain separation while allowing cross-graph queries through the dataset.

Vocabulary Loading for LLMs
For LLMs to understand partitioned data:

Vocabulary Identification: When preparing data for an LLM, identify all vocabularies/contexts used across the partitions.

Context Preprocessing: Preprocess contexts to simplify and make them more LLM-friendly (remove excessive URIs, focus on term meanings).

Relationship Explanation: Include metadata that explains the relationships between partitioned entities.

Namespace Disambiguation: Provide clear markers for which terms come from which vocabulary to avoid confusion.

Required Code Modifications
In _prepare_named_graph_structure:

Implement a more general partitioning algorithm that works for any context combination
Use our collision detection to guide the partitioning strategy
Add property distribution based on context compatibility
In add_jsonld_with_graph:

Enhance to better preserve context information
Ensure named graphs are properly utilized
Improve entity relationship tracking
In Query Methods:

Add methods to traverse references across partitions
Implement get_connected_entities(entity_id, max_depth=2)
Create reconstruct_logical_entity(entity_id, include_contexts=True)
For LLM Presentation:

Add format_for_llm(entity_ids, include_vocabulary_info=True)
Create vocabulary summary functions for context

## New Vocabulary Registry code

In [ ]:
# Update the EPCIS entry in VOCABULARY_REGISTRY
VOCABULARY_REGISTRY["epcis"] = {
    "uri": "https://ref.gs1.org/epcis/",
    "support_level": "direct",
    "context_location": "https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/epcis-context-root.jsonld",
    "access_pattern": "github_raw",
    "url_transformations": [
        {
            "pattern": r"https://gs1\.github\.io/EPCIS/(.+\.jsonld)",
            "replacement": r"https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/\1"
        }
    ],
    "uses_protection": True
}

In [ ]:
def transform_context_url(url):
    """Transform a context URL based on vocabulary registry rules."""
    import re
    
    # Check all vocabularies for URL transformation rules
    for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
        if "url_transformations" in vocab_info:
            for transform in vocab_info["url_transformations"]:
                pattern = transform["pattern"]
                replacement = transform["replacement"]
                
                # Apply the transformation if pattern matches
                transformed_url = re.sub(pattern, replacement, url)
                if transformed_url != url:
                    print(f"Transformed URL: {url} -> {transformed_url}")
                    return transformed_url
    
    # No transformation applied
    return url

In [ ]:
def create_vocab_aware_document_loader_enhanced(registry=None):
    """Create an enhanced document loader that handles URL transformations."""
    from pyld import jsonld
    
    # Use default registry if none provided
    registry = registry or VOCABULARY_REGISTRY
    
    # Get the default document loader to fall back to
    default_loader = jsonld.get_document_loader()
    
    # Create an in-memory cache for contexts
    context_cache = {}
    
    def enhanced_loader(url, options=None):
        """Custom document loader with URL transformation support."""
        options = options or {}
        
        # Apply URL transformations if needed
        transformed_url = transform_context_url(url)
        
        # Check if we have this in cache
        cache_key = transformed_url  # Use transformed URL as cache key
        if cache_key in context_cache:
            return context_cache[cache_key]
        
        # Rest of the loader logic remains similar to the original...
        # Check if this URL matches any known vocabulary
        for vocab_name, vocab_info in registry.items():
            vocab_uri = vocab_info["uri"]
            
            if transformed_url == vocab_uri or transformed_url.startswith(vocab_uri):
                support_level = vocab_info.get("support_level", "discover")
                
                # Handle based on support level
                if support_level == "direct":
                    result = _handle_direct_support(transformed_url, vocab_info, context_cache)
                    # Store in cache with original URL as well
                    if url != transformed_url:
                        context_cache[url] = result
                    return result
                elif support_level == "cache":
                    result = _handle_cache_support(transformed_url, vocab_info, context_cache, default_loader)
                    if url != transformed_url:
                        context_cache[url] = result
                    return result
                else:  # "discover"
                    result = _handle_discovery_support(transformed_url, default_loader, context_cache)
                    if url != transformed_url:
                        context_cache[url] = result
                    return result
        
        # No match found, use default loader with transformed URL
        try:
            result = default_loader(transformed_url, options)
            # Cache both URLs
            context_cache[cache_key] = result
            if url != transformed_url:
                context_cache[url] = result
            return result
        except Exception as e:
            # If transformed URL fails, try original as fallback
            if url != transformed_url:
                try:
                    result = default_loader(url, options)
                    context_cache[url] = result
                    return result
                except:
                    # Both failed, re-raise original error
                    raise e
            else:
                # No transformation was done, just raise the error
                raise
    
    return enhanced_loader

In [ ]:
def register_vocab_aware_loader():
    """Register our enhanced vocabulary-aware document loader with PyLD."""
    from pyld import jsonld
    
    # Create and register the enhanced loader
    loader = create_vocab_aware_document_loader_enhanced()
    jsonld.set_document_loader(loader)
    
    return loader

In [ ]:
def test_url_transformation():
    # Test with a URL that should be transformed
    gs1_url = "https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld"
    transformed = transform_context_url(gs1_url)
    print(f"Original: {gs1_url}")
    print(f"Transformed: {transformed}")
    
    # Test with a URL that shouldn't be transformed
    normal_url = "https://schema.org/docs/jsonldcontext.jsonld"
    non_transformed = transform_context_url(normal_url)
    print(f"\nOriginal: {normal_url}")
    print(f"After transform check: {non_transformed}")
    
    return transformed

In [ ]:
t=test_url_transformation()

Transformed URL: https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld -> https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/cbv-context-disposition.jsonld
Original: https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld
Transformed: https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/cbv-context-disposition.jsonld

Original: https://schema.org/docs/jsonldcontext.jsonld
After transform check: https://schema.org/docs/jsonldcontext.jsonld


## Code that needs name changes to integrate

In [ ]:
@patch
def _init_enhanced_indices(self:SemanticMemory):
    """Initialize enhanced indices for cross-graph querying."""
    # Add relationship tracking index if not present
    if "by_relationship" not in self.indices:
        self.indices["by_relationship"] = {
            "outgoing": {},  # entity_id -> {property: [target_ids]}
            "incoming": {}   # entity_id -> {property: [source_ids]}
        }
    
    # Add ID mapping index if not present
    if "id_mappings" not in self.indices:
        self.indices["id_mappings"] = {
            "original_to_generated": {},  # original_id -> [generated_ids]
            "generated_to_original": {}   # generated_id -> original_id
        }
    
    # Add container-based indices for JSON-LD 1.1
    if "containers" not in self.indices:
        self.indices["containers"] = {
            "by_id": {"@context": {"@container": "@index", "@index": "@id"}},
            "by_type": {"@context": {"@container": "@type"}}
        }

In [ ]:
@patch
def _track_entity_relationship(self:SemanticMemory, source_id, property_name, target_id):
    """Track relationships between entities for cross-graph navigation."""
    # Initialize if needed
    if "by_relationship" not in self.indices:
        self._init_enhanced_indices()
    
    # Track outgoing relationship
    if source_id not in self.indices["by_relationship"]["outgoing"]:
        self.indices["by_relationship"]["outgoing"][source_id] = {}
    
    if property_name not in self.indices["by_relationship"]["outgoing"][source_id]:
        self.indices["by_relationship"]["outgoing"][source_id][property_name] = []
    
    if target_id not in self.indices["by_relationship"]["outgoing"][source_id][property_name]:
        self.indices["by_relationship"]["outgoing"][source_id][property_name].append(target_id)
    
    # Track incoming relationship
    if target_id not in self.indices["by_relationship"]["incoming"]:
        self.indices["by_relationship"]["incoming"][target_id] = {}
    
    if property_name not in self.indices["by_relationship"]["incoming"][target_id]:
        self.indices["by_relationship"]["incoming"][target_id][property_name] = []
    
    if source_id not in self.indices["by_relationship"]["incoming"][target_id][property_name]:
        self.indices["by_relationship"]["incoming"][target_id][property_name].append(source_id)

In [ ]:
@patch
def query_by_id_enhanced(self:SemanticMemory, entity_id, follow_mappings=True):
    """Enhanced query that follows ID mappings and handles cross-graph references.
    
    Args:
        entity_id: ID of the entity to retrieve
        follow_mappings: Whether to follow ID mappings
        
    Returns:
        The entity if found, None otherwise
    """
    # Normalize the ID
    normalized_id = self.normalize_id(entity_id)
    
    # Try direct query first
    entity = self.query_by_id(normalized_id)
    if entity:
        return entity
    
    # Try with the original ID
    if normalized_id != entity_id:
        entity = self.query_by_id(entity_id)
        if entity:
            return entity
    
    # If not found and follow_mappings is enabled, check mappings
    if follow_mappings and "id_mappings" in self.indices:
        # Check if this is an original ID with generated mappings
        if normalized_id in self.indices["id_mappings"]["original_to_generated"]:
            generated_ids = self.indices["id_mappings"]["original_to_generated"][normalized_id]
            for gen_id in generated_ids:
                entity = self.query_by_id(gen_id)
                if entity:
                    return entity
        
        # Check if this is a generated ID with an original mapping
        if normalized_id in self.indices["id_mappings"]["generated_to_original"]:
            original_id = self.indices["id_mappings"]["generated_to_original"][normalized_id]
            entity = self.query_by_id(original_id)
            if entity:
                return entity
    
    # Try with/without trailing slash for URLs
    if normalized_id.startswith(('http://', 'https://')):
        alt_id = normalized_id + '/' if not normalized_id.endswith('/') else normalized_id[:-1]
        entity = self.query_by_id(alt_id)
        if entity:
            return entity
    
    return None

## Potential Memory patches

In [ ]:
@patch
def _prepare_named_graph_structure(self:SemanticMemory, data):
    """Prepare data as named graph structure for proper context isolation.
    
    This method implements a general approach for partitioning JSON-LD data with
    multiple contexts into a named graph structure. This prevents context collisions
    when contexts define conflicting terms or use @protected.
    
    The partitioning follows these principles:
    1. Each entity in the graph gets exactly one context
    2. Object properties become references to separate entities
    3. Contexts are distributed among entities in a rotating fashion
    4. Relationships are preserved through ID references
    
    Args:
        data: JSON-LD data to partition
        
    Returns:
        A JSON-LD document with @graph structure containing partitioned entities
    """
    import copy
    import uuid
    
    # Only proceed if we have a dict with a multi-context
    # (No need for graph separation otherwise)
    if not (isinstance(data, dict) and "@context" in data and 
            isinstance(data["@context"], list) and len(data["@context"]) >= 2):
        return data
    
    # Create the graph container to hold all partitioned entities
    graph_data = {"@graph": []}
    
    # Analyze contexts to determine partitioning strategy
    contexts = data["@context"]
    strategy = detect_collision_strategy_general(contexts)
    
    # Special case: Handle Verifiable Credential pattern
    # (This maintains backward compatibility with existing code)
    if ("credentialSubject" in data and isinstance(data["credentialSubject"], dict) and
        strategy and strategy.get("strategy") in ["graph_partition", "property_scoped"]):
        
        # Create the credential entity with first context
        credential = copy.deepcopy(data)
        subject = copy.deepcopy(data["credentialSubject"])
        
        # Ensure both have IDs for proper referencing
        if "@id" not in credential:
            credential["@id"] = f"urn:uuid:{uuid.uuid4()}"
        if "@id" not in subject:
            subject["@id"] = f"urn:uuid:{uuid.uuid4()}"
            
        # Use first context for credential, second for subject
        # This isolates the contexts to prevent collisions
        credential["@context"] = contexts[0]
        subject["@context"] = contexts[1]
        
        # Update credential to reference subject by ID instead of embedding
        credential["credentialSubject"] = {"@id": subject["@id"]}
        
        # Add both to the graph as separate entities
        graph_data["@graph"].append(credential)
        graph_data["@graph"].append(subject)
        
        return graph_data
    
    # General case: Analyze and partition the entity using a recursive approach
    # Start with the root entity using the first context
    root_entity = copy.deepcopy(data)
    root_entity["@context"] = contexts[0]  # Use first context for root
    
    # Ensure root has an ID for referencing
    if "@id" not in root_entity:
        root_entity["@id"] = f"urn:uuid:{uuid.uuid4()}"
    root_id = root_entity["@id"]
    
    # Track processed entities to avoid duplicates and circular references
    processed_entities = {root_id}
    
    def process_object_properties(entity, entity_context_idx=0):
        """Process object properties and extract them to separate entities.
        
        This recursive function:
        1. Identifies object properties in an entity
        2. Extracts them as separate entities with their own contexts
        3. Replaces the original property with an ID reference
        4. Handles both single objects and arrays of objects
        
        Args:
            entity: The entity to process
            entity_context_idx: Index of context to use for this entity
            
        Returns:
            The processed entity with references
        """
        if not isinstance(entity, dict):
            return entity
            
        # Get this entity's ID
        entity_id = entity.get("@id", None)
        if not entity_id:
            entity_id = f"urn:uuid:{uuid.uuid4()}"
            entity["@id"] = entity_id
            
        # Track as processed to avoid reprocessing
        processed_entities.add(entity_id)
        
        # Process each property in the entity
        for prop, value in list(entity.items()):
            # Skip JSON-LD keywords (they're not object properties)
            if prop.startswith("@"):
                continue
                
            # Handle object properties (dictionaries)
            if isinstance(value, dict) and len(value) > 1:
                # Check if this already has an ID
                if "@id" in value:
                    obj_id = value["@id"]
                else:
                    obj_id = f"urn:uuid:{uuid.uuid4()}"
                    value["@id"] = obj_id
                
                # Only process if not already processed (avoid cycles)
                if obj_id not in processed_entities:
                    # Create a new entity with the next context in rotation
                    # This distributes contexts among nested objects
                    next_context_idx = (entity_context_idx + 1) % len(contexts)
                    obj_entity = copy.deepcopy(value)
                    obj_entity["@context"] = contexts[next_context_idx]
                    
                    # Process nested properties in this object (recursion)
                    process_object_properties(obj_entity, next_context_idx)
                    
                    # Add to graph as a separate entity
                    graph_data["@graph"].append(obj_entity)
                
                # Replace with reference in parent entity
                # This maintains the relationship while separating contexts
                entity[prop] = {"@id": obj_id}
                
            # Handle arrays of objects
            elif isinstance(value, list):
                new_array = []
                for i, item in enumerate(value):
                    if isinstance(item, dict) and len(item) > 1:
                        # Check if this already has an ID
                        if "@id" in item:
                            item_id = item["@id"]
                        else:
                            item_id = f"urn:uuid:{uuid.uuid4()}"
                            item["@id"] = item_id
                        
                        # Only process if not already processed
                        if item_id not in processed_entities:
                            # Create a new entity with the next context in rotation
                            next_context_idx = (entity_context_idx + 1) % len(contexts)
                            item_entity = copy.deepcopy(item)
                            item_entity["@context"] = contexts[next_context_idx]
                            
                            # Process nested properties in this object (recursion)
                            process_object_properties(item_entity, next_context_idx)
                            
                            # Add to graph as a separate entity
                            graph_data["@graph"].append(item_entity)
                        
                        # Replace with reference
                        new_array.append({"@id": item_id})
                    else:
                        # Keep non-object items as is (strings, numbers, etc.)
                        new_array.append(item)
                
                # Update the array with references where appropriate
                entity[prop] = new_array
    
    # Start the recursive processing with the root entity
    process_object_properties(root_entity)
    
    # Add root entity to graph (must be done after processing to include all changes)
    graph_data["@graph"].append(root_entity)
    
    return graph_data

In [ ]:
@patch
def track_reference(self:SemanticMemory, source_id, property_name, target_id, is_forward_reference=False):
    """Track references between entities with enhanced ID handling.
    
    Args:
        source_id: ID of the source entity
        property_name: Name of the property establishing the reference
        target_id: ID of the target entity
        is_forward_reference: Whether this is a reference to an entity not yet processed
    """
    # Initialize if needed
    if "by_relationship" not in self.indices:
        self._init_enhanced_indices()
    
    # Normalize IDs
    source_id = self.normalize_id(source_id)
    target_id = self.normalize_id(target_id)
    
    if not source_id or not target_id:
        return
    
    # Check if source exists
    source_exists = source_id in self.indices["by_id"]
    if not source_exists:
        # Try to find source using ID mappings
        if "id_mappings" in self.indices:
            if source_id in self.indices["id_mappings"]["original_to_generated"]:
                for mapped_id in self.indices["id_mappings"]["original_to_generated"][source_id]:
                    if mapped_id in self.indices["by_id"]:
                        source_id = mapped_id
                        source_exists = True
                        break
            elif source_id in self.indices["id_mappings"]["generated_to_original"]:
                original = self.indices["id_mappings"]["generated_to_original"][source_id]
                if original in self.indices["by_id"]:
                    source_id = original
                    source_exists = True
    
    # Check if target exists, if not, treat as forward reference
    target_exists = target_id in self.indices["by_id"]
    if not target_exists and not is_forward_reference:
        # Try to find target using ID mappings
        if "id_mappings" in self.indices:
            if target_id in self.indices["id_mappings"]["original_to_generated"]:
                for mapped_id in self.indices["id_mappings"]["original_to_generated"][target_id]:
                    if mapped_id in self.indices["by_id"]:
                        target_id = mapped_id
                        target_exists = True
                        break
            elif target_id in self.indices["id_mappings"]["generated_to_original"]:
                original = self.indices["id_mappings"]["generated_to_original"][target_id]
                if original in self.indices["by_id"]:
                    target_id = original
                    target_exists = True
    
    if not target_exists and not is_forward_reference:
        is_forward_reference = True
    
    # Track in pending references if it's a forward reference
    if is_forward_reference:
        if "pending_references" not in self.indices:
            self.indices["pending_references"] = []
        
        # Add to pending references if not already there
        pending_ref = {
            "source_id": source_id,
            "property": property_name,
            "target_id": target_id
        }
        
        if pending_ref not in self.indices["pending_references"]:
            self.indices["pending_references"].append(pending_ref)
            print(f"Added pending reference: {source_id} -> {property_name} -> {target_id}")
        return
    
    # Both source and target exist, establish the relationship
    
    # Track outgoing relationship
    if source_id not in self.indices["by_relationship"]["outgoing"]:
        self.indices["by_relationship"]["outgoing"][source_id] = {}
    
    if property_name not in self.indices["by_relationship"]["outgoing"][source_id]:
        self.indices["by_relationship"]["outgoing"][source_id][property_name] = []
    
    if target_id not in self.indices["by_relationship"]["outgoing"][source_id][property_name]:
        self.indices["by_relationship"]["outgoing"][source_id][property_name].append(target_id)
    
    # Track incoming relationship
    if target_id not in self.indices["by_relationship"]["incoming"]:
        self.indices["by_relationship"]["incoming"][target_id] = {}
    
    if property_name not in self.indices["by_relationship"]["incoming"][target_id]:
        self.indices["by_relationship"]["incoming"][target_id][property_name] = []
    
    if source_id not in self.indices["by_relationship"]["incoming"][target_id][property_name]:
        self.indices["by_relationship"]["incoming"][target_id][property_name].append(source_id)

In [ ]:
@patch
def resolve_pending_references(self:SemanticMemory):
    """Resolve any pending forward references after processing all entities."""
    if "pending_references" not in self.indices:
        return
        
    resolved_count = 0
    still_pending = []
    
    for ref in self.indices["pending_references"]:
        source_id = ref["source_id"]
        property_name = ref["property"]
        target_id = ref["target_id"]
        
        # Check if target exists now
        if target_id in self.indices["by_id"] or self.query_by_id_enhanced(target_id):
            # Target exists, establish the relationship
            self.track_reference(source_id, property_name, target_id)
            resolved_count += 1
        else:
            # Target still doesn't exist, keep in pending
            still_pending.append(ref)
    
    # Update pending references list
    self.indices["pending_references"] = still_pending
    
    return resolved_count

In [ ]:
@patch
def detect_context_compatibility(self:SemanticMemory, context1, context2):
    """Determine if two contexts are compatible or will cause collisions.
    
    Args:
        context1: First context (dict, string, or list)
        context2: Second context (dict, string, or list)
        
    Returns:
        tuple: (compatible, reason)
            - compatible: Boolean indicating if contexts can be used together
            - reason: Explanation of compatibility or collision
    """
    # Handle string contexts (URLs)
    if isinstance(context1, str) and isinstance(context2, str):
        # Same context is always compatible with itself
        if context1 == context2:
            return True, "Same context"
        
        # Check known vocabulary combinations
        ctx1_vocab = None
        ctx2_vocab = None
        
        # Try to match with known vocabularies
        for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
            if context1 == vocab_info["uri"] or context1.startswith(vocab_info["uri"]):
                ctx1_vocab = vocab_name
            if context2 == vocab_info["uri"] or context2.startswith(vocab_info["uri"]):
                ctx2_vocab = vocab_name
        
        # If both are known vocabularies, check compatibility
        if ctx1_vocab and ctx2_vocab:
            # Check if we have a known strategy for this pair
            vocab_pair = tuple(sorted([ctx1_vocab, ctx2_vocab]))
            if vocab_pair in COLLISION_STRATEGIES:
                strategy = COLLISION_STRATEGIES[vocab_pair]["strategy"]
                return strategy != "graph_partition", f"Known vocabulary pair: {strategy}"
            
            # Check if either uses protection
            uses_protection = (VOCABULARY_REGISTRY.get(ctx1_vocab, {}).get("uses_protection", False) or 
                              VOCABULARY_REGISTRY.get(ctx2_vocab, {}).get("uses_protection", False))
            if uses_protection:
                return False, "Protected vocabulary detected"
        
        # Default to incompatible for unknown combinations
        return False, "Unknown vocabulary combination"
    
    # Handle dict contexts
    if isinstance(context1, dict) and isinstance(context2, dict):
        # Check for @protected flag
        if context1.get("@protected", False) or context2.get("@protected", False):
            return False, "Protected context detected"
        
        # Check for term collisions
        collisions = []
        for term in context1:
            if term in context2 and term not in ["@version", "@vocab"]:
                if context1[term] != context2[term]:
                    collisions.append(term)
        
        if collisions:
            return False, f"Term collisions: {', '.join(collisions)}"
        
        # Check base vocabularies
        vocab1 = context1.get("@vocab")
        vocab2 = context2.get("@vocab")
        if vocab1 and vocab2 and vocab1 != vocab2:
            return False, "Different base vocabularies"
    
    # Handle list contexts recursively
    if isinstance(context1, list) or isinstance(context2, list):
        # Convert single contexts to lists for uniform handling
        ctx1_list = context1 if isinstance(context1, list) else [context1]
        ctx2_list = context2 if isinstance(context2, list) else [context2]
        
        # Check each combination
        for c1 in ctx1_list:
            for c2 in ctx2_list:
                compatible, reason = self.detect_context_compatibility(c1, c2)
                if not compatible:
                    return False, reason
    
    # If we get here, the contexts are compatible
    return True, "Compatible contexts"

In [ ]:
@patch
def _prepare_named_graph_structure(self:SemanticMemory, data):
    """Improved method to prepare data as named graph structure with strict context isolation.
    
    This version ensures:
    1. Each entity gets exactly one context
    2. Contexts are never mixed
    3. Relationships are preserved through ID references
    4. No keyword collisions occur
    
    Args:
        data: JSON-LD data to partition
        
    Returns:
        A JSON-LD document with @graph structure
    """
    import copy
    import uuid
    
    # Initialize enhanced indices
    self._init_enhanced_indices()
    
    # Only proceed if we have a dict
    if not isinstance(data, dict):
        return data
    
    # Create the graph container
    graph_data = {"@graph": []}
    contexts = data.get("@context", {})
    
    # For list contexts, check compatibility between them
    context_compatibility = {}
    if isinstance(contexts, list) and len(contexts) > 1:
        # Check each pair of contexts for compatibility
        for i, ctx1 in enumerate(contexts):
            for j, ctx2 in enumerate(contexts):
                if i != j:
                    compatible, reason = self.detect_context_compatibility(ctx1, ctx2)
                    context_compatibility[(i, j)] = (compatible, reason)
    
    # Handle the VC pattern specifically
    if ("credentialSubject" in data and isinstance(data["credentialSubject"], dict) and
        (isinstance(contexts, list) and len(contexts) >= 2)):
        
        # Create the credential entity with first context
        credential = copy.deepcopy(data)
        subject = copy.deepcopy(data["credentialSubject"])
        
        # Normalize and preserve IDs
        credential_id = credential.get("@id", credential.get("id"))
        if not credential_id:
            credential_id = f"urn:uuid:{uuid.uuid4()}"
        else:
            credential_id = self.normalize_id(credential_id)
            
        subject_id = subject.get("@id", subject.get("id"))
        if not subject_id:
            subject_id = f"urn:uuid:{uuid.uuid4()}"
        else:
            subject_id = self.normalize_id(subject_id)
        
        # Normalize IDs to @id format
        credential["@id"] = credential_id
        subject["@id"] = subject_id
        
        # Handle ID mapping
        if "id" in credential and credential["id"] != credential_id:
            self._track_id_mapping(self.normalize_id(credential["id"]), credential_id)
            
        if "id" in subject and subject["id"] != subject_id:
            self._track_id_mapping(self.normalize_id(subject["id"]), subject_id)
            
        # CRITICAL: Use ONLY one context for each entity
        credential["@context"] = contexts[0] if isinstance(contexts, list) and len(contexts) > 0 else contexts
        subject["@context"] = contexts[1] if isinstance(contexts, list) and len(contexts) > 1 else contexts
        
        # Remove any nested contexts that might cause conflicts
        if "@context" in credential["credentialSubject"]:
            del credential["credentialSubject"]["@context"]
        
        # Update credential to reference subject by ID only
        credential["credentialSubject"] = {"@id": subject_id}
        
        # Track this relationship
        self.track_reference(credential_id, "credentialSubject", subject_id)
        
        # Add both to the graph
        graph_data["@graph"].append(credential)
        graph_data["@graph"].append(subject)
        
        return graph_data
    
    # General case for other structures with multiple contexts
    # Start with the root entity
    root_entity = copy.deepcopy(data)
    
    # Normalize and preserve root ID
    root_id = root_entity.get("@id", root_entity.get("id"))
    if not root_id:
        root_id = f"urn:uuid:{uuid.uuid4()}"
    else:
        root_id = self.normalize_id(root_id)
    
    # Normalize to @id format
    root_entity["@id"] = root_id
    
    # Handle ID mapping
    if "id" in root_entity and root_entity["id"] != root_id:
        self._track_id_mapping(self.normalize_id(root_entity["id"]), root_id)
    
    # Assign first context to root
    if isinstance(contexts, list) and len(contexts) > 0:
        root_entity["@context"] = contexts[0]
    else:
        root_entity["@context"] = contexts
    
    # Process nested objects
    processed_entities = {root_id}
    
    def process_entity_tree(entity, context_idx=0, parent_id=None, parent_property=None):
        """Process an entity and its nested objects recursively."""
        if not isinstance(entity, dict):
            return entity
        
        # Get or generate entity ID
        entity_id = entity.get("@id", entity.get("id"))
        original_id = entity_id
        
        if not entity_id:
            entity_id = f"urn:uuid:{uuid.uuid4()}"
        else:
            entity_id = self.normalize_id(entity_id)
            
        entity["@id"] = entity_id
        
        # Track ID mapping if needed
        if original_id and original_id != entity_id:
            self._track_id_mapping(self.normalize_id(original_id), entity_id)
        
        # Track relationship with parent if applicable
        if parent_id and parent_property:
            self.track_reference(parent_id, parent_property, entity_id)
        
        # Skip if already processed
        if entity_id in processed_entities:
            return entity_id
        
        processed_entities.add(entity_id)
        
        # CRITICAL: Assign exactly one context
        if isinstance(contexts, list) and len(contexts) > 0:
            # Find a context that's compatible with the parent's context
            if parent_id and parent_property:
                # Try to find a compatible context
                parent_context_idx = context_idx
                for i, ctx in enumerate(contexts):
                    if (parent_context_idx, i) in context_compatibility:
                        compatible, _ = context_compatibility[(parent_context_idx, i)]
                        if compatible:
                            context_idx = i
                            break
            
            entity["@context"] = contexts[context_idx % len(contexts)]
        else:
            entity["@context"] = contexts
        
        # Remove any nested contexts
        for prop, value in entity.items():
            if prop.startswith("@"):
                continue
            if isinstance(value, dict) and "@context" in value:
                del value["@context"]
        
        # Process nested objects
        for prop, value in list(entity.items()):
            if prop.startswith("@"):
                continue
                
            # Handle object properties
            if isinstance(value, dict) and len(value) > 1:
                # Find next context index
                next_idx = (context_idx + 1) % len(contexts) if isinstance(contexts, list) and len(contexts) > 0 else 0
                
                nested_id = process_entity_tree(
                    value, 
                    next_idx,
                    entity_id,
                    prop
                )
                entity[prop] = {"@id": nested_id}
                
            # Handle arrays of objects
            elif isinstance(value, list):
                for i, item in enumerate(value):
                    if isinstance(item, dict) and len(item) > 1:
                        # Find next context index
                        next_idx = (context_idx + 1) % len(contexts) if isinstance(contexts, list) and len(contexts) > 0 else 0
                        
                        item_id = process_entity_tree(
                            item,
                            next_idx,
                            entity_id,
                            prop
                        )
                        value[i] = {"@id": item_id}
        
        # Add to graph
        graph_data["@graph"].append(entity)
        return entity_id
    
    # Start processing from root
    process_entity_tree(root_entity)
    
    # Resolve any pending references
    self.resolve_pending_references()
    
    return graph_data

In [ ]:
@patch
def add_jsonld_with_graph(self:SemanticMemory, data, _recursion_depth=0):
    """Add JSON-LD data with @graph structure with improved entity processing.
    
    Args:
        data: JSON-LD data with @graph structure
        _recursion_depth: Internal parameter to track recursion
        
    Returns:
        List of processed entities
    """
    import copy
    import uuid
    import json
    from rdflib import Graph, URIRef, Literal

    # Recursion guard
    if _recursion_depth >= 3:
        print(f"WARNING: Maximum recursion depth reached in add_jsonld_with_graph ({_recursion_depth})")
        raise ValueError("Maximum recursion depth exceeded in add_jsonld_with_graph")
    
    # Initialize enhanced indices if needed
    self._init_enhanced_indices()
    
    # If data doesn't have @graph, use regular add_jsonld
    if not isinstance(data, dict) or '@graph' not in data:
        return self.add_jsonld(data, _recursion_depth=_recursion_depth+1)
    
    results = []
    
    # Extract shared context if present
    shared_context = data.get('@context')
    
    # Process each entity individually with its own context
    for i, entity in enumerate(data['@graph']):
        try:
            # Skip if this is just a reference
            if isinstance(entity, dict) and len(entity) == 1 and '@id' in entity:
                continue
                
            # Apply shared context if entity doesn't have one
            entity_copy = copy.deepcopy(entity)
            if shared_context and '@context' not in entity_copy:
                entity_copy['@context'] = shared_context
            
            # Apply strict property normalization
            entity_copy = self._normalize_properties(entity_copy)
            
            # Ensure entity has an ID
            if '@id' not in entity_copy:
                entity_id = f"urn:uuid:{uuid.uuid4()}"
                entity_copy['@id'] = entity_id
            else:
                entity_id = entity_copy['@id']
                
            # Normalize ID
            original_id = entity_id
            entity_id = self.normalize_id(entity_id)
            entity_copy['@id'] = entity_id
            
            # Track ID mapping if original was different
            if original_id != entity_id:
                self._track_id_mapping(original_id, entity_id)
            
            print(f"Processing graph entity {i+1}/{len(data['@graph'])}: {entity_id}")
            
            # Store original data
            self.original_data[entity_id] = entity_copy
            
            # Register context if present
            if '@context' in entity_copy:
                context_id = self._register_contexts(entity_copy)
                
                # Track context association
                if entity_id not in self.entity_contexts:
                    self.entity_contexts[entity_id] = {
                        "primary_context": None,
                        "property_contexts": {}
                    }
                self.entity_contexts[entity_id]["primary_context"] = context_id
                self.context_registry[context_id]["used_by"].add(entity_id)
            
            # Try to expand the entity
            try:
                expanded = jsonld.expand(entity_copy)
                
                # Update indices with expanded data
                self._update_indices(expanded)
                self._update_indices_with_labels(expanded)
                
                # Add to RDF graph
                g = Graph()
                g.parse(data=json.dumps(expanded), format='json-ld')
                self.default_graph += g
                
                # Track relationships
                for prop, value in entity_copy.items():
                    if prop.startswith('@'):
                        continue
                        
                    if isinstance(value, dict) and '@id' in value:
                        target_id = value['@id']
                        self.track_reference(entity_id, prop, target_id)
                    elif isinstance(value, list):
                        for item in value:
                            if isinstance(item, dict) and '@id' in item:
                                target_id = item['@id']
                                self.track_reference(entity_id, prop, target_id)
                
                results.append(expanded[0] if isinstance(expanded, list) and expanded else expanded)
                print(f"Successfully added entity {entity_id} to indices and graph")
                
            except Exception as expand_error:
                # Fallback to direct indexing if expansion fails
                print(f"Expansion failed for {entity_id}: {str(expand_error)}")
                print("Using direct indexing as fallback")
                
                # Add to by_id index
                self.indices["by_id"][entity_id] = entity_copy
                
                # Add to by_type index if type is present
                if '@type' in entity_copy:
                    types = entity_copy['@type'] if isinstance(entity_copy['@type'], list) else [entity_copy['@type']]
                    for type_uri in types:
                        if type_uri not in self.indices["by_type"]:
                            self.indices["by_type"][type_uri] = []
                        self.indices["by_type"][type_uri].append(entity_copy)
                
                # Add relationships
                for prop, value in entity_copy.items():
                    if prop.startswith('@'):
                        continue
                        
                    if isinstance(value, dict) and '@id' in value:
                        target_id = value['@id']
                        self.track_reference(entity_id, prop, target_id)
                    elif isinstance(value, list):
                        for item in value:
                            if isinstance(item, dict) and '@id' in item:
                                target_id = item['@id']
                                self.track_reference(entity_id, prop, target_id)
                
                results.append(entity_copy)
                print(f"Successfully added entity {entity_id} with direct indexing")
            
        except Exception as e:
            print(f"Error processing graph entity {i+1}: {e}")
            # Continue with other entities
    
    # Resolve any pending references
    self.resolve_pending_references()
    
    return results

In [ ]:
@patch
def add_jsonld(self:SemanticMemory, data, context=None, _recursion_depth=0):
    """Add JSON-LD data to semantic memory with improved ID handling.
    
    This version ensures:
    1. Proper ID normalization and mapping
    2. Context isolation through graph partitioning when needed
    3. Consistent relationship tracking
    
    Args:
        data: JSON-LD data to add
        context: Optional context to apply or merge with existing context
        _recursion_depth: Internal parameter to track recursion (do not set manually)
        
    Returns:
        The expanded JSON-LD data after processing
    """
    
    # Check recursion depth to prevent infinite loops
    if _recursion_depth >= 3:
        print(f"WARNING: Maximum recursion depth reached ({_recursion_depth})")
        raise ValueError("Maximum recursion depth exceeded in add_jsonld")
    
    # Initialize enhanced indices if needed
    if "by_relationship" not in self.indices:
        self._init_enhanced_indices()
    
    # Make a deep copy to avoid modifying the original
    data_copy = copy.deepcopy(data)
    
    # If context is provided, merge it with the data
    if context:
        if isinstance(data_copy, dict):
            if "@context" in data_copy:
                # Merge contexts
                if isinstance(data_copy["@context"], list):
                    if isinstance(context, list):
                        data_copy["@context"] = context + data_copy["@context"]
                    else:
                        data_copy["@context"] = [context] + data_copy["@context"]
                else:
                    if isinstance(context, list):
                        data_copy["@context"] = context + [data_copy["@context"]]
                    else:
                        data_copy["@context"] = [context, data_copy["@context"]]
            else:
                data_copy["@context"] = context
    
    # Handle ID normalization and mapping
    if isinstance(data_copy, dict):
        # Save original ID if present
        original_id = None
        if "id" in data_copy:
            original_id = data_copy["id"]
        elif "@id" in data_copy:
            original_id = data_copy["@id"]
        
        # Apply strict property normalization
        data_copy = self._normalize_properties(data_copy)
        
        # Ensure ID exists and is normalized
        if "@id" not in data_copy:
            data_copy["@id"] = f"urn:uuid:{uuid.uuid4()}"
        
        # Normalize the ID
        normalized_id = self.normalize_id(data_copy["@id"])
        
        # Track mapping if ID changed during normalization
        if original_id and normalized_id != original_id:
            self._track_id_mapping(original_id, normalized_id)
            data_copy["@id"] = normalized_id
        
        # Store original data with normalized ID
        self.original_data[normalized_id] = data_copy
        
        # Check if we need graph partitioning
        if self._needs_graph_separation(data_copy):
            # Use our enhanced graph partitioning
            graph_data = self._prepare_named_graph_structure(data_copy)
            # Process the graph structure
            return self.add_jsonld_with_graph(graph_data, _recursion_depth=_recursion_depth+1)
    
    # For data that doesn't need partitioning, proceed with normal processing
    try:
        # Register contexts if present
        if isinstance(data_copy, dict) and "@context" in data_copy:
            context_id = self._register_contexts(data_copy)
            
            # Store entity-context association
            entity_id = data_copy.get("@id")
            if entity_id and context_id:
                if entity_id not in self.entity_contexts:
                    self.entity_contexts[entity_id] = {
                        "primary_context": None,
                        "property_contexts": {}
                    }
                self.entity_contexts[entity_id]["primary_context"] = context_id
                self.context_registry[context_id]["used_by"].add(entity_id)
        
        # Track relationships in the data
        if isinstance(data_copy, dict):
            entity_id = data_copy.get("@id")
            if entity_id:
                # Process each property for relationships
                for prop, value in data_copy.items():
                    if prop.startswith('@'):
                        continue
                        
                    # Handle object references
                    if isinstance(value, dict) and '@id' in value:
                        target_id = value['@id']
                        self.track_reference(entity_id, prop, target_id)
                    # Handle arrays of references
                    elif isinstance(value, list):
                        for item in value:
                            if isinstance(item, dict) and '@id' in item:
                                target_id = item['@id']
                                self.track_reference(entity_id, prop, target_id)
        
        # Expand the data for our indices
        expanded = jsonld.expand(data_copy)
        
        # Update indices
        self._update_indices(expanded)
        self._update_indices_with_labels(expanded)
        
        # Add to RDF graph
        g = Graph()
        g.parse(data=json.dumps(expanded), format='json-ld')
        self.default_graph += g
        
        # Resolve any pending references
        resolved_count = self.resolve_pending_references()
        if resolved_count:
            print(f"Resolved {resolved_count} references, {len(self.indices.get('pending_references', []))} still pending")
        
        return expanded
        
    except Exception as e:
        # If expansion fails, try graph partitioning as fallback
        if isinstance(data_copy, dict) and _recursion_depth < 2:
            try:
                print(f"Expansion failed with {str(e)}, trying graph partitioning as fallback")
                graph_data = self._prepare_named_graph_structure(data_copy)
                # Process with add_jsonld_with_graph but increment recursion depth
                return self.add_jsonld_with_graph(graph_data, _recursion_depth=_recursion_depth+1)
            except Exception as graph_error:
                print(f"Graph partitioning fallback also failed: {str(graph_error)}")
                
        # Re-raise the original error if fallback also fails
        raise

## Debugging code

In [ ]:
def debug_context_collisions():
    """Debug context collision issues with step-by-step tests."""
    print("=== Context Collision Debugging ===\n")
    
    # Initialize memory
    memory = SemanticMemory()
    
    # Test 1: Load and use VC context alone
    print("Test 1: VC context alone")
    vc_only = {
        "@context": "https://www.w3.org/ns/credentials/v2",
        "@id": "urn:test:vc1",
        "@type": ["VerifiableCredential"],
        "issuer": "did:example:issuer1"
    }
    
    try:
        expanded_vc = jsonld.expand(vc_only)
        print("✓ Successfully expanded VC-only document")
        # Print the expanded structure to see how terms are expanded
        print(f"Term expansions in VC context:")
        print(f"  @id → {next(iter(expanded_vc))['@id']}")
        print(f"  @type → {next(iter(expanded_vc))['@type']}")
    except Exception as e:
        print(f"✗ VC expansion failed: {e}")
    
    # Test 2: Load and use EPCIS context alone
    print("\nTest 2: EPCIS context alone")
    epcis_only = {
        "@context": "https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/epcis-context-root.jsonld",
        "@id": "urn:test:epcis1",
        "@type": "ObjectEvent",
        "eventTime": "2023-11-15T13:45:00Z"
    }
    
    try:
        expanded_epcis = jsonld.expand(epcis_only)
        print("✓ Successfully expanded EPCIS-only document")
        print(f"Term expansions in EPCIS context:")
        print(f"  @id → {next(iter(expanded_epcis))['@id']}")
        print(f"  @type → {next(iter(expanded_epcis))['@type'][0]}")
    except Exception as e:
        print(f"✗ EPCIS expansion failed: {e}")
    
    # Test 3: Print the actual context content to examine structure
    print("\nTest 3: Examining context structures")
    
    try:
        # Use document loader to get contexts
        vc_ctx_doc = jsonld.get_document_loader()("https://www.w3.org/ns/credentials/v2")
        vc_ctx = vc_ctx_doc["document"].get("@context", {})
        
        epcis_ctx_doc = jsonld.get_document_loader()("https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/epcis-context-root.jsonld")
        epcis_ctx = epcis_ctx_doc["document"].get("@context", {})
        
        # Check for protected flag
        print(f"VC context has @protected: {'@protected' in vc_ctx and vc_ctx['@protected']}")
        print(f"EPCIS context has @protected: {'@protected' in epcis_ctx and epcis_ctx['@protected']}")
        
        # Check specific term definitions
        print("\nTerm definitions in both contexts:")
        for term in ["id", "type"]:
            print(f"'{term}' in VC context: {term in vc_ctx}")
            if term in vc_ctx:
                print(f"  Definition: {vc_ctx[term]}")
                
            print(f"'{term}' in EPCIS context: {term in epcis_ctx}")
            if term in epcis_ctx:
                print(f"  Definition: {epcis_ctx[term]}")
    except Exception as e:
        print(f"Context examination failed: {e}")
    
    # Test 4: Try simplified combined context with just the colliding terms
    print("\nTest 4: Testing simplified combined context")
    
    simplified = {
        "@context": [
            {"@protected": True, "id": "@id", "type": "@type"},  # VC style
            {"id": "@id", "type": "@type"}  # EPCIS style
        ],
        "@id": "urn:test:combined1",
        "type": "TestEntity"
    }
    
    try:
        expanded_simplified = jsonld.expand(simplified)
        print("✓ Successfully expanded simplified combined document")
    except Exception as e:
        print(f"✗ Simplified combined expansion failed: {e}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error details: {getattr(e, 'details', 'No details')}")
    
    return "Debugging complete"

In [ ]:
debug_context_collisions()

=== Context Collision Debugging ===

Test 1: VC context alone
✓ Successfully expanded VC-only document
Term expansions in VC context:
  @id → urn:test:vc1
  @type → ['https://www.w3.org/2018/credentials#VerifiableCredential']

Test 2: EPCIS context alone
Transformed URL: https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld -> https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/cbv-context-disposition.jsonld
✓ Successfully expanded EPCIS-only document
Term expansions in EPCIS context:
  @id → urn:test:epcis1
  @type → https://ref.gs1.org/epcis/ObjectEvent

Test 3: Examining context structures
VC context has @protected: True
EPCIS context has @protected: True

Term definitions in both contexts:
'id' in VC context: True
  Definition: @id
'id' in EPCIS context: True
  Definition: @id
'type' in VC context: True
  Definition: @type
'type' in EPCIS context: True
  Definition: @type

Test 4: Testing simplified combined context
✓ Successfully expanded simplified combined do

'Debugging complete'

In [ ]:
@patch
def normalize_id(self:SemanticMemory, id_value):
    """Normalize different ID formats for consistent handling.
    
    This handles:
    - URL normalization (trailing slashes)
    - Basic string normalization
    
    Args:
        id_value: Original ID in any format
        
    Returns:
        Normalized ID string
    """
    if not id_value:
        return None
        
    # Convert to string if needed
    id_str = str(id_value)
    
    # Handle URL normalization
    if id_str.startswith(('http://', 'https://')):
        # Remove trailing slash if present
        if id_str.endswith('/') and len(id_str) > 8:  # Don't remove from http:// itself
            id_str = id_str[:-1]
    
    return id_str

In [ ]:
@patch
def normalize_properties(self:SemanticMemory, data):
    """Normalize JSON-LD properties for consistent handling.
    
    This normalizes:
    - 'id' → '@id'
    - 'type' → '@type'
    
    Args:
        data: JSON-LD data to normalize
        
    Returns:
        Normalized data
    """
    if not isinstance(data, dict):
        return data
        
    result = copy.deepcopy(data)
    
    # Handle id → @id
    if "id" in result:
        if "@id" not in result:
            result["@id"] = result["id"]
        del result["id"]
        
    # Handle type → @type
    if "type" in result:
        if "@type" not in result:
            result["@type"] = result["type"]
        del result["type"]
        
    # Recursively normalize nested objects
    for key, value in list(result.items()):
        if key.startswith('@'):
            continue
            
        if isinstance(value, dict):
            result[key] = self.normalize_properties(value)
        elif isinstance(value, list):
            result[key] = [
                self.normalize_properties(item) if isinstance(item, dict) else item
                for item in value
            ]
    
    return result

Here's how we can solve this step by step:

1. Simplify ID Management
The debugging shows that simple ID cases work fine, but our comprehensive test still fails. This suggests:

We need consistent ID normalization across all methods
ID tracking needs to happen at the right stage in processing
2. Clarify Responsibility Boundaries
_normalize_properties_strict: Should ONLY handle normalization of properties
_prepare_named_graph_structure: Should handle partitioning AND preserve IDs
add_jsonld_with_graph: Should handle adding the partitioned entities to memory
3. Simplify Our Approach
Let's reduce complexity by:

Using one canonical way to normalize IDs and properties
Ensuring relationship tracking happens in only one place
Making sure each function has a single responsibility
4. Step-by-Step Implementation Plan
Create a single, reliable ID normalization function
Simplify the property normalization to just handle id → @id and type → @type
Update the graph partitioning to use these functions consistently
Ensure every entity added to the graph is properly indexed

Looking through our dialog, here are the functions we can simplify or remove to streamline our approach:

Functions to Remove/Consolidate:
ID Management Functions:

_track_id_mapping and normalize_and_track_id can be consolidated into a single normalize_id function
The complex ID mapping indices are overkill - we can use a simpler approach
Property Normalization:

_normalize_properties_with_tracking and _normalize_properties_strict are redundant
We only need one simple normalization function
Relationship Tracking:

track_reference and _track_entity_relationship do similar things
The pending references system adds complexity without clear benefit
Query Functions:

query_by_id_enhanced can be merged into the main query_by_id method
Overly Complex Components:
Enhanced Indices:

The whole _init_enhanced_indices structure is more complex than needed
We can use a simpler index structure
Context Compatibility Detection:

detect_context_compatibility is complex and not essential for the core functionality
Multiple Graph Partitioning Methods:

We have multiple versions of _prepare_named_graph_structure that can be consolidated
Simplification Strategy:
Start with a clean, simple ID normalization
Create one clear property normalization function
Simplify the graph partitioning to handle just the essentials
Make query methods straightforward
This will give us a more maintainable codebase that still handles the core requirements effectively.

In [ ]:
def debug_partitioning_process():
    """Debug the partitioning and expansion process step by step."""
    print("=== Partitioning Process Debugging ===\n")
    
    # Create a simple test document
    test_doc = {
        "@context": [
            "https://www.w3.org/ns/credentials/v2",
            "https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/epcis-context-root.jsonld"
        ],
        "id": "urn:test:credential:123",
        "type": ["VerifiableCredential"],
        "credentialSubject": {
            "id": "urn:test:subject:456",
            "type": "ObjectEvent"
        }
    }
    
    print("1. Original document structure:")
    print(json.dumps(test_doc, indent=2)[:500])
    
    # Step 2: Create the partitioned structure
    memory = SemanticMemory()
    
    print("\n2. Creating partitioned structure")
    try:
        graph_data = memory._prepare_named_graph_structure(test_doc)
        print("✓ Successfully created graph structure")
        print(f"Graph has {len(graph_data.get('@graph', []))} entities")
        
        # Print each entity's structure
        for i, entity in enumerate(graph_data.get('@graph', [])):
            print(f"\nEntity {i+1}:")
            print(json.dumps(entity, indent=2)[:300])
    except Exception as e:
        print(f"✗ Graph partitioning failed: {e}")
        return
    
    # Step 3: Try to expand each entity separately
    print("\n3. Expanding each entity separately")
    for i, entity in enumerate(graph_data.get('@graph', [])):
        print(f"\nExpanding entity {i+1}:")
        try:
            expanded = jsonld.expand(entity)
            print(f"✓ Successfully expanded entity {i+1}")
        except Exception as e:
            print(f"✗ Expansion failed for entity {i+1}: {e}")
            print(f"Entity structure: {json.dumps(entity, indent=2)[:200]}")
    
    # Step 4: Try alternative partitioning approach
    print("\n4. Testing alternative partitioning approach")
    
    # Create a more simplified graph structure
    simple_graph = {
        "@graph": [
            {
                "@context": "https://www.w3.org/ns/credentials/v2",
                "@id": "urn:test:credential:123",
                "@type": ["VerifiableCredential"],
                "credentialSubject": {"@id": "urn:test:subject:456"}
            },
            {
                "@context": {"@vocab": "https://ref.gs1.org/epcis/"},
                "@id": "urn:test:subject:456",
                "@type": "ObjectEvent"
            }
        ]
    }
    
    print("Testing expansion of simplified graph:")
    try:
        expanded_graph = jsonld.expand(simple_graph)
        print("✓ Successfully expanded simplified graph")
    except Exception as e:
        print(f"✗ Expansion failed for simplified graph: {e}")
    
    return "Debugging complete"

In [ ]:
debug_partitioning_process()

=== Partitioning Process Debugging ===

1. Original document structure:
{
  "@context": [
    "https://www.w3.org/ns/credentials/v2",
    "https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/epcis-context-root.jsonld"
  ],
  "id": "urn:test:credential:123",
  "type": [
    "VerifiableCredential"
  ],
  "credentialSubject": {
    "id": "urn:test:subject:456",
    "type": "ObjectEvent"
  }
}

2. Creating partitioned structure
Added pending reference: urn:test:credential:123 -> credentialSubject -> urn:test:subject:456
✓ Successfully created graph structure
Graph has 2 entities

Entity 1:
{
  "@context": "https://www.w3.org/ns/credentials/v2",
  "id": "urn:test:credential:123",
  "type": [
    "VerifiableCredential"
  ],
  "credentialSubject": {
    "@id": "urn:test:subject:456"
  },
  "@id": "urn:test:credential:123"
}

Entity 2:
{
  "id": "urn:test:subject:456",
  "type": "ObjectEvent",
  "@id": "urn:test:subject:456",
  "@context": "https://raw.githubusercontent.com/gs1/

'Debugging complete'

In [ ]:
def debug_document_retrieval():
    """Debug the document retrieval issue with detailed tracing."""
    # Initialize fresh memory instance
    memory = SemanticMemory()
    print("=== Document Retrieval Debugging ===\n")
    
    # Create test document with specific ID
    test_doc = {
        "@context": "https://schema.org/",
        "id": "urn:test:document:1",
        "type": "Document",
        "name": "Test Document"
    }
    
    # Step 1: Check normalization
    print("1. Testing ID normalization:")
    original_id = test_doc["id"]
    normalized_id = memory.normalize_id(original_id)
    print(f"Original ID: {original_id}")
    print(f"Normalized ID: {normalized_id}")
    
    # Step 2: Add document to memory
    print("\n2. Adding document to memory:")
    try:
        expanded = memory.add_jsonld(test_doc)
        print("✓ Document added successfully")
    except Exception as e:
        print(f"✗ Failed to add document: {e}")
        return
    
    # Step 3: Examine indices to see what's there
    print("\n3. Examining indices:")
    print(f"Keys in by_id index: {list(memory.indices['by_id'].keys())}")
    
    if "id_mappings" in memory.indices:
        print("\nID mappings:")
        print(f"original_to_generated: {memory.indices['id_mappings']['original_to_generated']}")
        print(f"generated_to_original: {memory.indices['id_mappings']['generated_to_original']}")
    
    # Step 4: Test queries with different methods
    print("\n4. Testing different query methods:")
    
    # Direct query by original ID
    result1 = memory.query_by_id(original_id)
    print(f"query_by_id('{original_id}'): {result1 is not None}")
    
    # Direct query by normalized ID
    result2 = memory.query_by_id(normalized_id)
    print(f"query_by_id('{normalized_id}'): {result2 is not None}")
    
    # Enhanced query that should check mappings
    result3 = memory.query_by_id_enhanced(original_id)
    print(f"query_by_id_enhanced('{original_id}'): {result3 is not None}")
    
    # Check what's in the original data store
    print("\n5. Checking original_data store:")
    original_data_keys = list(memory.original_data.keys())
    print(f"Keys in original_data: {original_data_keys}")
    
    return memory

In [ ]:
mem = debug_document_retrieval()

=== Document Retrieval Debugging ===

1. Testing ID normalization:
Original ID: urn:test:document:1
Normalized ID: urn:test:document:1

2. Adding document to memory:
✗ Failed to add document: name 'copy' is not defined


**AI Prompt**

Have a look through this dialog and could you tell me what we're trying to do in this dialog?

**AI Response**

Based on the dialog, it looks like you're working on enhancing a semantic memory system in the cogitarelink package, with a focus on improving how it handles JSON-LD data with multiple vocabularies and contexts.

Specifically, you're:

1. Debugging and fixing issues with context collisions in JSON-LD documents that use multiple vocabularies (like VC and EPCIS)

2. Implementing improved graph partitioning techniques to properly separate entities with different contexts while maintaining their relationships

3. Adding better ID normalization and tracking to ensure consistent entity references across partitioned graphs

4. Creating enhanced query mechanisms that can work effectively across partitioned data

5. Adding URL transformation capabilities to handle edge cases with vocabulary contexts (particularly for EPCIS)

The dialog shows several debugging attempts to identify why JSON-LD expansion fails with certain context combinations, followed by implementation of more sophisticated approaches to context separation. You're essentially making the system more robust for handling complex semantic data while preserving the relationships needed for LLM understanding.

Would you like me to focus on any particular aspect of this work in more detail?

In [ ]:
def test_semantic_memory_new_structure():
    """Tests for semantic memory with proposed new structure functions"""
    import uuid, json
    from pathlib import Path
    
    # Setup
    cache_dir = Path("./test_cache")
    cache_dir.mkdir(exist_ok=True)
    memory = SemanticMemory(cache_dir=cache_dir)
    
    print("=== Testing New Semantic Memory Structure ===\n")
    
    # Test 1: Normalization functions
    print("Test 1: Entity normalization")
    test_entity = {
        "id": "urn:test:entity:1",
        "type": "Person",
        "name": "Test Person",
        "details": {"id": "urn:test:details:1", "type": "ContactInfo"}
    }
    
    try:
        # Test normalization
        normalized = memory._normalize_entity(test_entity)
        assert "@id" in normalized and "id" not in normalized, "ID not normalized"
        assert "@type" in normalized and "type" not in normalized, "Type not normalized"
        assert "@id" in normalized["details"] and "id" not in normalized["details"], "Nested ID not normalized"
        print("✓ Entity normalization working correctly")
    except Exception as e:
        print(f"✗ Entity normalization failed: {e}")
    
    # Test 2: Context registration
    print("\nTest 2: Context registration")
    test_context = {
        "@version": 1.1,
        "@vocab": "https://schema.org/",
        "nested": {
            "@id": "https://example.org/nested",
            "@context": {
                "@vocab": "https://example.org/vocab/"
            }
        }
    }
    
    try:
        # Register context
        ctx_id = memory._register_context(test_context)
        assert ctx_id is not None, "Context registration failed"
        assert ctx_id in memory.context_registry, "Context not found in registry"
        
        # Check scoped context extraction
        scoped_contexts = memory.context_registry[ctx_id]["scoped_contexts"]
        assert "nested" in scoped_contexts, "Scoped context not extracted"
        print("✓ Context registration with scoped contexts working")
    except Exception as e:
        print(f"✗ Context registration failed: {e}")
    
    # Test 3: Process entity
    print("\nTest 3: Entity processing")
    basic_entity = {
        "@context": "https://schema.org/",
        "@id": "urn:test:entity:basic",
        "@type": "Person",
        "name": "Basic Entity"
    }
    
    try:
        # Process single entity
        result = memory._process_entity(basic_entity)
        assert result is not None, "Entity processing failed"
        
        # Check if it was added to indices
        entity = memory.query_by_id("urn:test:entity:basic")
        assert entity is not None, "Entity not found in indices"
        print("✓ Entity processing working")
    except Exception as e:
        print(f"✗ Entity processing failed: {e}")
    
    # Test 4: Add method as single entry point
    print("\nTest 4: Add method as single entry point")
    complex_entity = {
        "@context": [
            "https://schema.org/",
            "https://www.w3.org/ns/activitystreams"
        ],
        "@id": "urn:test:complex:1",
        "@type": ["Person", "Actor"],
        "name": "Complex Entity",
        "outbox": {
            "@id": "urn:test:outbox:1",
            "@type": "OrderedCollection",
            "totalItems": 10
        }
    }
    
    try:
        # Add using the single entry point
        memory.add(complex_entity)
        
        # Check both entities were added
        entity1 = memory.query_by_id("urn:test:complex:1")
        entity2 = memory.query_by_id("urn:test:outbox:1")
        assert entity1 is not None, "Main entity not found"
        assert entity2 is not None, "Nested entity not found"
        print("✓ Add method working for complex entities")
    except Exception as e:
        print(f"✗ Add method failed: {e}")
    
    # Test 5: Graph partitioning and processing
    print("\nTest 5: Graph partitioning and processing")
    vc_data = {
        "@context": [
            "https://www.w3.org/ns/credentials/v2",
            "https://schema.org/"
        ],
        "@id": "urn:uuid:" + str(uuid.uuid4()),
        "@type": ["VerifiableCredential"],
        "issuer": "https://example.org/issuer",
        "credentialSubject": {
            "@id": "urn:test:subject:1",
            "@type": "Person",
            "name": "Credential Subject"
        }
    }
    
    try:
        # Test if it needs partitioning
        needs_partition = memory._needs_graph_partition(vc_data)
        assert needs_partition, "Failed to detect need for partitioning"
        
        # Create graph partition
        graph_data = memory._create_graph_partition(vc_data)
        assert "@graph" in graph_data, "Graph partitioning failed"
        assert len(graph_data["@graph"]) == 2, f"Expected 2 entities, got {len(graph_data['@graph'])}"
        
        # Process the graph
        results = memory._process_graph(graph_data)
        assert len(results) == 2, "Graph processing returned wrong number of entities"
        
        # Check that both entities were added
        credential = memory.query_by_id(vc_data["@id"])
        subject = memory.query_by_id("urn:test:subject:1")
        assert credential is not None, "Credential not found"
        assert subject is not None, "Subject not found"
        print("✓ Graph partitioning and processing working")
    except Exception as e:
        print(f"✗ Graph partitioning failed: {e}")
    
    # Test 6: Query and relationship traversal
    print("\nTest 6: Query and relationship traversal")
    try:
        # Create entities with relationships for testing
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:a",
            "@type": "Person",
            "name": "Person A",
            "knows": [
                {"@id": "urn:test:person:b"},
                {"@id": "urn:test:person:c"}
            ]
        })
        
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:b",
            "@type": "Person",
            "name": "Person B"
        })
        
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:c",
            "@type": "Person",
            "name": "Person C",
            "knows": {"@id": "urn:test:person:d"}
        })
        
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:d",
            "@type": "Person",
            "name": "Person D"
        })
        
        # Test connected entities retrieval
        connected = memory.get_connected_entities("urn:test:person:a", max_depth=2)
        assert len(connected) >= 3, f"Expected at least 3 connected entities, got {len(connected)}"
        assert "urn:test:person:d" in connected, "Failed to traverse to depth 2"
        print("✓ Relationship traversal working correctly")
    except Exception as e:
        print(f"✗ Relationship traversal failed: {e}")
    
    # Test 7: Index updating 
    print("\nTest 7: Index updating")
    try:
        # Create entity for index testing
        index_test = {
            "@context": {
                "@vocab": "http://schema.org/",
                "rdfs": "http://www.w3.org/2000/01/rdf-schema#"
            },
            "@id": "urn:test:index:1",
            "@type": "Product",
            "name": "Indexed Product",
            "rdfs:label": "Unique Test Label",
            "description": "Product with special keywords for testing"
        }
        
        memory.add(index_test)
        
        # Test type indexing
        type_index = memory.indices["by_type"].get("http://schema.org/Product", [])
        assert any(e.get("@id") == "urn:test:index:1" for e in type_index), "Type indexing failed"
        
        # Test label indexing
        if hasattr(memory, "_index_labels_and_descriptions"):
            # Force indexing to ensure it gets processed
            entity = memory.query_by_id("urn:test:index:1")
            memory._index_labels_and_descriptions(entity)
            
            assert "by_label" in memory.indices, "Label index not created"
            label_matches = memory.indices["by_label"].get("Unique Test Label", [])
            assert len(label_matches) > 0, "Label index not working"
        
        print("✓ Index updating working correctly")
    except Exception as e:
        print(f"✗ Index updating failed: {e}")
    
    # Test 8: JSON-LD 1.1 features
    print("\nTest 8: JSON-LD 1.1 features")
    try:
        # Test with JSON-LD 1.1 containers and scoped contexts
        jsonld11_test = {
            "@context": {
                "@version": 1.1,
                "schema": "https://schema.org/",
                "team": {
                    "@id": "schema:team",
                    "@context": {
                        "@vocab": "https://schema.org/",
                        "members": {"@container": "@set"}
                    }
                },
                "categories": {
                    "@id": "schema:category",
                    "@container": "@index"
                }
            },
            "@id": "urn:test:jsonld11:1",
            "@type": "schema:Organization",
            "schema:name": "JSON-LD 1.1 Test",
            "team": {
                "@id": "urn:test:team:1",
                "name": "Core Team",
                "members": [
                    {"@id": "urn:test:person:a"},
                    {"@id": "urn:test:person:b"}
                ]
            },
            "categories": {
                "primary": "Software",
                "secondary": "Research"
            }
        }
        
        memory.add(jsonld11_test)
        org = memory.query_by_id("urn:test:jsonld11:1")
        team = memory.query_by_id("urn:test:team:1")
        
        assert org is not None, "Organization not added"
        assert team is not None, "Team not added"
        print("✓ JSON-LD 1.1 features working")
    except Exception as e:
        print(f"✗ JSON-LD 1.1 test failed: {e}")
    
    # Test 9: Context-aware retrieval
    print("\nTest 9: Context-aware retrieval")
    try:
        # Create entity with scoped context for testing
        scoped_test = {
            "@context": {
                "@version": 1.1,
                "@vocab": "https://schema.org/",
                "details": {
                    "@id": "https://example.org/details",
                    "@context": {
                        "@vocab": "https://example.org/terms/",
                        "code": "identifier"
                    }
                }
            },
            "@id": "urn:test:scoped:1",
            "@type": "Event",
            "name": "Scoped Context Test",
            "details": {
                "@id": "urn:test:details:1",
                "name": "Event Details",
                "code": "EVENT-123"
            }
        }
        
        memory.add(scoped_test)
        
        # Retrieve with context preserved
        entity = memory.retrieve_in_context("urn:test:scoped:1")
        
        assert entity is not None, "Failed to retrieve entity with context"
        assert "@context" in entity, "Context not preserved"
        details_ctx = entity.get("@context", {}).get("details", {}).get("@context")
        assert details_ctx is not None, "Scoped context not preserved"
        print("✓ Context-aware retrieval working")
    except Exception as e:
        print(f"✗ Context-aware retrieval failed: {e}")
    
    # Final summary
    print("\n=== Final Stats ===")
    print(f"Total entities: {len(memory.indices['by_id'])}")
    print(f"Total contexts: {len(memory.context_registry)}")
    print(f"Entity types: {len(memory.indices['by_type'])}")
    
    return memory

In [ ]:
#| export
class SemanticMemory:
    def __init__(self, cache_dir=None):
        # Core storage
        self.dataset = Dataset()
        self.default_graph = self.dataset.graph(URIRef("urn:x-arq:DefaultGraph"))
        self.original_data = {}
        
        # Simple, clear indices
        self.indices = {
            "by_id": {},
            "by_type": {},
            "by_label": {},
            "by_keyword": {}
        }
        
        # Context management
        self.contexts = {}
        self.context_registry = {}
        
        # Client for external resources
        self.client = httpx.Client(follow_redirects=True)
        
        # Optional cache
        self.cache_dir = cache_dir
        if cache_dir: Path(cache_dir).mkdir(exist_ok=True)

## Let's create a unified way to add data to memory:

In [ ]:
#| export
@patch 
def add(self:SemanticMemory, data, context=None):
    """Single entry point for adding JSON-LD data to memory"""
    import copy, uuid
    
    # Handle context merging if provided
    if context and isinstance(data, dict):
        data = copy.deepcopy(data)
        if "@context" not in data: data["@context"] = context
        elif isinstance(data["@context"], list): data["@context"] = [context] + data["@context"]
        else: data["@context"] = [context, data["@context"]]
    
    # Choose processing path based on structure
    if isinstance(data, dict) and "@graph" in data: return self._process_graph(data)
    if self._needs_graph_partition(data): return self._process_graph(self._create_graph_partition(data))
    return self._process_entity(data)

## Core processing methods:

In [ ]:
#| export
@patch
def _needs_graph_partition(self:SemanticMemory, data):
    """Determine if data needs graph partitioning"""
    if not isinstance(data, dict): return False
    if "@context" in data and isinstance(data["@context"], list) and len(data["@context"]) > 1: return True
    if "credentialSubject" in data and isinstance(data["credentialSubject"], dict): return True
    return False

In [ ]:
#| export
@patch
def _create_graph_partition(self:SemanticMemory, data):
    """Convert document into graph with separate entities for each context"""
    import copy, uuid
    
    graph = {"@graph": []}
    contexts = data.get("@context", [])
    if not isinstance(contexts, list): contexts = [contexts]
    
    # Create normalized root entity
    root = self._normalize_entity(copy.deepcopy(data))
    root_id = root.get("@id") or f"urn:uuid:{uuid.uuid4()}"
    root["@id"] = root_id
    root["@context"] = contexts[0] if contexts else data.get("@context")
    
    # Track processed entities to avoid cycles
    processed = {root_id}
    
    # Recursive function to extract and partition nested entities
    def extract_nested(entity, ctx_idx=0, parent_id=None, link_prop=None):
        """Extract nested objects as separate entities with their own contexts"""
        if not isinstance(entity, dict): return entity
        
        # Skip if already processed
        entity_id = entity.get("@id") or f"urn:uuid:{uuid.uuid4()}"
        entity["@id"] = entity_id
        if entity_id in processed: return {"@id": entity_id}
        processed.add(entity_id)
        
        # Add to graph if not already there
        if entity not in graph["@graph"]: graph["@graph"].append(entity)
        
        # Set context for this entity based on position in context list
        next_idx = (ctx_idx + 1) % len(contexts) if contexts else 0
        if "@context" not in entity: entity["@context"] = contexts[next_idx] if contexts else root["@context"]
        
        # Process each property for nested objects
        for prop, val in list(entity.items()):
            if prop.startswith("@"): continue
            
            # Handle single object
            if isinstance(val, dict) and len(val) > 1:
                nested = self._normalize_entity(copy.deepcopy(val))
                nested_id = extract_nested(nested, next_idx, entity_id, prop)["@id"]
                entity[prop] = {"@id": nested_id}
            
            # Handle arrays of objects
            elif isinstance(val, list):
                for i, item in enumerate(val):
                    if isinstance(item, dict) and len(item) > 1:
                        nested = self._normalize_entity(copy.deepcopy(item))
                        nested_id = extract_nested(nested, next_idx, entity_id, prop)["@id"]
                        val[i] = {"@id": nested_id}
        
        return entity
    
    # Start extraction from root entity
    extract_nested(root)
    
    return graph

In [ ]:
#| export
@patch
def _process_graph(self:SemanticMemory, data):
    """Process a graph structure with multiple entities"""
    if not isinstance(data, dict) or "@graph" not in data: return None
    
    results = []
    shared_ctx = data.get("@context")
    
    # Process each entity
    for entity in data["@graph"]:
        # Skip bare references
        if isinstance(entity, dict) and len(entity) == 1 and "@id" in entity: continue
        
        # Apply shared context if needed
        if shared_ctx and isinstance(entity, dict) and "@context" not in entity:
            entity = dict(entity, **{"@context": shared_ctx})
        
        # Process entity
        result = self._process_entity(entity)
        if result: results.append(result)
    
    return results

In [ ]:
#| export
@patch
def _process_entity(self:SemanticMemory, data):
    """Process a single entity, adding it to indices and graph"""
    import copy, json, uuid
    from rdflib import Graph
    
    if not isinstance(data, dict): return None
    
    # Normalize the entity
    entity = self._normalize_entity(copy.deepcopy(data))
    
    # Ensure ID
    if "@id" not in entity: entity["@id"] = f"urn:uuid:{uuid.uuid4()}"
    entity_id = entity["@id"]
    
    # Store original data
    self.original_data[entity_id] = entity
    
    # Register context and track associations
    if "@context" in entity:
        ctx_id = self._register_context(entity["@context"])
        if ctx_id:
            ctx_info = self.context_registry.setdefault(ctx_id, {"used_by": set()})
            ctx_info["used_by"].add(entity_id)
    
    # Try expanding and indexing
    try:
        expanded = jsonld.expand(entity)
        
        # Update indices
        self._update_indices(expanded)
        
        # Add to RDF graph
        g = Graph()
        g.parse(data=json.dumps(expanded), format="json-ld")
        self.default_graph += g
        
        return expanded
    except Exception as e:
        # Fallback to direct indexing
        print(f"JSON-LD expansion failed: {e}")
        self.indices["by_id"][entity_id] = entity
        
        if "@type" in entity:
            types = [entity["@type"]] if not isinstance(entity["@type"], list) else entity["@type"]
            for t in types:
                if t not in self.indices["by_type"]: self.indices["by_type"][t] = []
                self.indices["by_type"][t].append(entity)
        
        return entity

## Context and Entity Normalization

In [ ]:
#| export
@patch
def _normalize_entity(self:SemanticMemory, data):
    """Normalize entity properties for consistent processing"""
    if not isinstance(data, dict): return data
    
    result = data.copy()
    
    # Normalize id → @id
    if "id" in result:
        if "@id" not in result: result["@id"] = result["id"]
        del result["id"]
    
    # Normalize type → @type
    if "type" in result:
        if "@type" not in result: result["@type"] = result["type"]
        del result["type"]
    
    # Recursively normalize nested objects
    for k, v in list(result.items()):
        if k.startswith("@"): continue
        if isinstance(v, dict): result[k] = self._normalize_entity(v)
        elif isinstance(v, list): result[k] = [self._normalize_entity(i) if isinstance(i, dict) else i for i in v]
    
    return result

In [ ]:
#| export
@patch
def _register_context(self:SemanticMemory, context):
    """Register context and extract scoped contexts"""
    import json, hashlib
    
    # Generate stable ID for context
    if isinstance(context, str): ctx_id = context
    elif isinstance(context, list): ctx_id = "ctx:" + "_".join(self._register_context(c) for c in context)
    elif isinstance(context, dict):
        ctx_str = json.dumps(context, sort_keys=True)
        ctx_id = "ctx:" + hashlib.md5(ctx_str.encode()).hexdigest()
    else: return None
    
    # Skip if already registered
    if ctx_id in self.context_registry: return ctx_id
    
    # Register new context
    self.context_registry[ctx_id] = {
        "context": context,
        "scoped_contexts": {},
        "used_by": set()
    }
    
    # Extract scoped contexts if present
    if isinstance(context, dict):
        for term, defn in context.items():
            if isinstance(defn, dict) and "@context" in defn:
                # Register the scoped context
                scoped_id = self._register_context(defn["@context"])
                if scoped_id: self.context_registry[ctx_id]["scoped_contexts"][term] = scoped_id
    
    return ctx_id

In [ ]:
def test_register_context():
    memory = SemanticMemory()
    
    # Test simple string context
    ctx_id1 = memory._register_context("https://schema.org/")
    assert ctx_id1 == "https://schema.org/"
    assert ctx_id1 in memory.context_registry
    
    # Test object context
    obj_context = {
        "@vocab": "https://schema.org/",
        "name": "https://schema.org/name"
    }
    ctx_id2 = memory._register_context(obj_context)
    assert ctx_id2.startswith("ctx:")
    assert ctx_id2 in memory.context_registry
    assert memory.context_registry[ctx_id2]["context"] == obj_context
    
    # Test scoped context
    scoped_context = {
        "@vocab": "https://schema.org/",
        "nested": {
            "@id": "https://example.org/nested",
            "@context": {
                "@vocab": "https://example.org/"
            }
        }
    }
    ctx_id3 = memory._register_context(scoped_context)
    assert ctx_id3 in memory.context_registry
    assert "nested" in memory.context_registry[ctx_id3]["scoped_contexts"]
    
    # Test list context
    list_context = ["https://schema.org/", "https://www.w3.org/ns/activitystreams"]
    ctx_id4 = memory._register_context(list_context)
    assert ctx_id4.startswith("ctx:")
    assert ctx_id4 in memory.context_registry
    
    print("✓ Context registration tests passed")
    return memory.context_registry

In [ ]:
test_register_context()

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[1], line 1
----> 1 test_register_context()

Cell In[1], line 5, in test_register_context()
      2 memory = SemanticMemory()
      4 # Test simple string context
----> 5 ctx_id1 = memory._register_context("https://schema.org/")
      6 assert ctx_id1 == "https://schema.org/"
      7 assert ctx_id1 in memory.context_registry

AttributeError: 'SemanticMemory' object has no attribute '_register_context'


AttributeError: 'SemanticMemory' object has no attribute '_register_context'

In [ ]:
def test_normalize_entity():
    memory = SemanticMemory()
    
    # Test basic normalization
    test_entity = {
        "id": "urn:test:entity:1",
        "type": "Person",
        "name": "Test Person"
    }
    
    normalized = memory._normalize_entity(test_entity)
    
    # Check id/type normalization
    assert "@id" in normalized and "id" not in normalized
    assert "@type" in normalized and "type" not in normalized
    assert normalized["@id"] == "urn:test:entity:1"
    assert normalized["@type"] == "Person"
    
    # Test nested normalization
    nested_entity = {
        "id": "urn:test:entity:2",
        "type": "Organization",
        "name": "Test Org",
        "contact": {
            "id": "urn:test:contact:1",
            "type": "ContactPoint",
            "email": "test@example.com"
        }
    }
    
    normalized_nested = memory._normalize_entity(nested_entity)
    
    # Check nested normalization
    assert "@id" in normalized_nested["contact"]
    assert "id" not in normalized_nested["contact"]
    assert "@type" in normalized_nested["contact"]
    assert "type" not in normalized_nested["contact"]
    
    print("✓ Entity normalization tests passed")
    return normalized_nested

In [ ]:
norm = test_normalize_entity()

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[1], line 1
----> 1 norm = test_normalize_entity()

Cell In[1], line 11, in test_normalize_entity()
      4 # Test basic normalization
      5 test_entity = {
      6     "id": "urn:test:entity:1",
      7     "type": "Person",
      8     "name": "Test Person"
      9 }
---> 11 normalized = memory._normalize_entity(test_entity)
     13 # Check id/type normalization
     14 assert "@id" in normalized and "id" not in normalized

AttributeError: 'SemanticMemory' object has no attribute '_normalize_entity'


AttributeError: 'SemanticMemory' object has no attribute '_normalize_entity'

**AI Prompt**

Okay, so it looks like we have an error in our semantic memory test function.

**AI Response**

It looks like we're encountering an error because the `_normalize_entity` method isn't being properly defined on the `SemanticMemory` class. Even though we defined the method with the `@patch` decorator, it seems the patching hasn't taken effect yet.

Let's implement the method directly to fix this issue:

```python
@patch
def _normalize_entity(self:SemanticMemory, data):
    """Normalize entity properties for consistent processing"""
    if not isinstance(data, dict): return data
    
    result = data.copy()
    
    # Normalize id → @id
    if "id" in result:
        if "@id" not in result: result["@id"] = result["id"]
        del result["id"]
    
    # Normalize type → @type
    if "type" in result:
        if "@type" not in result: result["@type"] = result["type"]
        del result["type"]
    
    # Recursively normalize nested objects
    for k, v in list(result.items()):
        if k.startswith("@"): continue
        if isinstance(v, dict): result[k] = self._normalize_entity(v)
        elif isinstance(v, list): result[k] = [self._normalize_entity(i) if isinstance(i, dict) else i for i in v]
    
    return result
```

Let's run this again to see if the patching takes effect. If this doesn't work, we might need to ensure that the patch is being applied correctly or modify our approach to defining the method.

## Query Methods

In [ ]:
#| export
@patch
def query_by_id(self:SemanticMemory, entity_id):
    """Get entity by ID with improved retrieval"""
    # Try direct lookup
    if entity_id in self.indices["by_id"]: return self.indices["by_id"][entity_id]
    
    # Try with/without trailing slash for URLs
    if entity_id.startswith(('http://', 'https://')):
        alt_id = entity_id + '/' if not entity_id.endswith('/') else entity_id[:-1]
        if alt_id in self.indices["by_id"]: return self.indices["by_id"][alt_id]
    
    # Try RDF query as fallback
    entity_uri = URIRef(entity_id)
    results = {}
    
    for s, p, o in self.default_graph.triples((entity_uri, None, None)):
        pred = str(p)
        if pred not in results: results[pred] = []
        
        if isinstance(o, URIRef): results[pred].append({"@id": str(o)})
        elif isinstance(o, Literal): results[pred].append({"@value": str(o)})
    
    return results if results else None

In [ ]:
#| export
@patch
def get_connected_entities(self:SemanticMemory, entity_id, max_depth=1):
    """Get entity and all connected entities up to max_depth"""
    if max_depth < 0: return {}
    
    # Get the starting entity
    entity = self.query_by_id(entity_id)
    if not entity: return {}
    
    result = {entity_id: entity}
    
    # Find all referenced entities
    if max_depth > 0:
        for prop, values in entity.items():
            if prop.startswith("@"): continue
            
            for val in values if isinstance(values, list) else [values]:
                if isinstance(val, dict) and "@id" in val:
                    ref_id = val["@id"]
                    # Recursively get connected entities
                    connected = self.get_connected_entities(ref_id, max_depth-1)
                    result.update(connected)
    
    return result

In [ ]:
#| export
@patch
def query_by_type(self:SemanticMemory, type_uri):
    """Get all entities of a specific type"""
    if type_uri in self.indices["by_type"]: return self.indices["by_type"][type_uri]
    
    # Fallback to RDF query
    type_uri_ref = URIRef(type_uri)
    entities = []
    
    for s in self.default_graph.subjects(RDF.type, type_uri_ref):
        entity = {"@id": str(s), "@type": type_uri}
        
        for p, o in self.default_graph.predicate_objects(s):
            pred = str(p)
            if pred == str(RDF.type): continue
            
            if pred not in entity: entity[pred] = []
            
            if isinstance(o, URIRef): entity[pred].append({"@id": str(o)})
            elif isinstance(o, Literal): entity[pred].append({"@value": str(o)})
        
        entities.append(entity)
    
    return entities

## Update indexing

In [ ]:
#| export
@patch
def _update_indices(self:SemanticMemory, expanded_data):
    """Update indices with expanded JSON-LD data"""
    # Handle both single entity and array cases
    nodes = expanded_data if isinstance(expanded_data, list) else [expanded_data]
    
    for node in nodes:
        if not isinstance(node, dict): continue
        
        # Get node ID
        node_id = node.get('@id')
        if not node_id: continue
        
        # Update by_id index
        self.indices["by_id"][node_id] = node
        
        # Update by_type index
        types = node.get('@type', [])
        if not isinstance(types, list): types = [types]
        
        for type_uri in types:
            if type_uri not in self.indices["by_type"]: self.indices["by_type"][type_uri] = []
            if node not in self.indices["by_type"][type_uri]: 
                self.indices["by_type"][type_uri].append(node)
        
        # Update label and description indices
        self._index_labels_and_descriptions(node)

In [ ]:
#| export
@patch
def _index_labels_and_descriptions(self:SemanticMemory, node):
    """Index entity by labels and textual content for better search"""
    if not isinstance(node, dict): return
    
    # Common label properties
    label_props = [
        "http://www.w3.org/2000/01/rdf-schema#label",
        "http://schema.org/name",
        "http://purl.org/dc/terms/title",
        "http://xmlns.com/foaf/0.1/name"
    ]
    
    # Common description properties
    desc_props = [
        "http://www.w3.org/2000/01/rdf-schema#comment",
        "http://schema.org/description",
        "http://purl.org/dc/terms/description",
        "http://purl.org/dc/elements/1.1/description"
    ]
    
    node_id = node.get('@id')
    if not node_id: return
    
    # Extract labels for label index
    for prop in label_props:
        if prop in node:
            for value in node[prop]:
                if isinstance(value, dict) and '@value' in value:
                    label = value['@value']
                    if label not in self.indices["by_label"]: self.indices["by_label"][label] = []
                    if {'@id': node_id} not in self.indices["by_label"][label]:
                        self.indices["by_label"][label].append({'@id': node_id})
    
    # Extract text from descriptions for keyword index
    for prop in desc_props:
        if prop in node:
            for value in node[prop]:
                if isinstance(value, dict) and '@value' in value:
                    text = value['@value']
                    # Simple keyword extraction - split and filter
                    keywords = [w.lower() for w in text.split() if len(w) > 3]
                    for keyword in keywords:
                        if keyword not in self.indices["by_keyword"]: self.indices["by_keyword"][keyword] = []
                        if {'@id': node_id} not in self.indices["by_keyword"][keyword]:
                            self.indices["by_keyword"][keyword].append({'@id': node_id})

## Retrieval and Search Functions

In [ ]:
#| export
@patch
def retrieve_in_context(self:SemanticMemory, entity_id):
    """Retrieve entity with original context structure preserved"""
    # Check if we have the original data
    if entity_id not in self.original_data: return None
    
    # Get the original data with context
    entity = self.original_data[entity_id]
    
    # Look up context registry entries for this entity
    ctx_ids = []
    for ctx_id, ctx_info in self.context_registry.items():
        if entity_id in ctx_info.get("used_by", set()):
            ctx_ids.append(ctx_id)
    
    # Ensure scoped contexts are properly included
    import copy
    result = copy.deepcopy(entity)
    
    # Process any context with scoped contexts
    for ctx_id in ctx_ids:
        ctx_info = self.context_registry[ctx_id]
        
        # Check if this context has scoped contexts
        if ctx_info["scoped_contexts"] and isinstance(result.get("@context"), dict):
            # Ensure all scoped contexts are included
            for term, scoped_id in ctx_info["scoped_contexts"].items():
                if term in result["@context"] and isinstance(result["@context"][term], dict):
                    # Get the scoped context definition
                    scoped_ctx = self.context_registry.get(scoped_id, {}).get("context")
                    if scoped_ctx:
                        # Add back the scoped context
                        result["@context"][term]["@context"] = scoped_ctx
    
    return result

In [ ]:
#| export
@patch
def search(self:SemanticMemory, query_text, limit=10):
    """Search memory for entities matching the query text"""
    results = []
    
    # First check exact label matches
    if "by_label" in self.indices:
        for label, entities in self.indices["by_label"].items():
            if query_text.lower() in label.lower():
                for entity_ref in entities:
                    if "@id" in entity_ref:
                        entity = self.query_by_id(entity_ref["@id"])
                        if entity and entity not in results:
                            results.append(entity)
                            if len(results) >= limit: return results
    
    # Then check keyword matches
    if "by_keyword" in self.indices:
        # Simple tokenization
        tokens = query_text.lower().split()
        for token in tokens:
            if len(token) > 3 and token in self.indices["by_keyword"]:
                for entity_ref in self.indices["by_keyword"][token]:
                    if "@id" in entity_ref:
                        entity = self.query_by_id(entity_ref["@id"])
                        if entity and entity not in results:
                            results.append(entity)
                            if len(results) >= limit: return results
    
    # If still not enough results, try content search
    if len(results) < limit:
        for entity_id, entity in self.indices["by_id"].items():
            entity_str = str(entity).lower()
            if query_text.lower() in entity_str and entity not in results:
                results.append(entity)
                if len(results) >= limit: return results
    
    return results

## Integration points for Vocabulary Tools

In [ ]:
#| export
@patch
def use_vocabulary(self:SemanticMemory, vocab_name):
    """Load and register a vocabulary for use in this memory instance"""
    # This would call VocabTools functions but not depend on VocabTools internals
    context = load_context_for_vocabulary(vocab_name)
    return self._register_context(context)

In [ ]:
@patch
def compact_with_vocabulary(self:SemanticMemory, entity_id, vocab_name):
    """Retrieve an entity and compact it using a specific vocabulary"""
    entity = self.query_by_id(entity_id)
    if not entity: return None
    return compact_entity_with_vocabulary(entity, vocab_name)

**AI Prompt**

Okay, so I want to use the Solveit approach. Now we've defined a whole bunch of stuff, but we haven't really atomically tested any of it. So even though we have comprehensive behavior tests from end to end, we don't know if our actual functions do the things that they should do.

**AI Response**

You're right - we've created a complete redesign of the semantic memory system with a cleaner structure, but we haven't tested the individual functions to verify they work as expected. Let's create some atomic tests for the key functions we've defined.

Let's start with a test for the core entity normalization function:

```python
def test_normalize_entity():
    memory = SemanticMemory()
    
    # Test basic normalization
    test_entity = {
        "id": "urn:test:entity:1",
        "type": "Person",
        "name": "Test Person"
    }
    
    normalized = memory._normalize_entity(test_entity)
    
    # Check id/type normalization
    assert "@id" in normalized and "id" not in normalized
    assert "@type" in normalized and "type" not in normalized
    assert normalized["@id"] == "urn:test:entity:1"
    assert normalized["@type"] == "Person"
    
    # Test nested normalization
    nested_entity = {
        "id": "urn:test:entity:2",
        "type": "Organization",
        "name": "Test Org",
        "contact": {
            "id": "urn:test:contact:1",
            "type": "ContactPoint",
            "email": "test@example.com"
        }
    }
    
    normalized_nested = memory._normalize_entity(nested_entity)
    
    # Check nested normalization
    assert "@id" in normalized_nested["contact"]
    assert "id" not in normalized_nested["contact"]
    assert "@type" in normalized_nested["contact"]
    assert "type" not in normalized_nested["contact"]
    
    print("✓ Entity normalization tests passed")
    return normalized_nested
```

Now let's test the context registration function:

```python
def test_register_context():
    memory = SemanticMemory()
    
    # Test simple string context
    ctx_id1 = memory._register_context("https://schema.org/")
    assert ctx_id1 == "https://schema.org/"
    assert ctx_id1 in memory.context_registry
    
    # Test object context
    obj_context = {
        "@vocab": "https://schema.org/",
        "name": "https://schema.org/name"
    }
    ctx_id2 = memory._register_context(obj_context)
    assert ctx_id2.startswith("ctx:")
    assert ctx_id2 in memory.context_registry
    assert memory.context_registry[ctx_id2]["context"] == obj_context
    
    # Test scoped context
    scoped_context = {
        "@vocab": "https://schema.org/",
        "nested": {
            "@id": "https://example.org/nested",
            "@context": {
                "@vocab": "https://example.org/"
            }
        }
    }
    ctx_id3 = memory._register_context(scoped_context)
    assert ctx_id3 in memory.context_registry
    assert "nested" in memory.context_registry[ctx_id3]["scoped_contexts"]
    
    # Test list context
    list_context = ["https://schema.org/", "https://www.w3.org/ns/activitystreams"]
    ctx_id4 = memory._register_context(list_context)
    assert ctx_id4.startswith("ctx:")
    assert ctx_id4 in memory.context_registry
    
    print("✓ Context registration tests passed")
    return memory.context_registry
```

Let's test the graph partitioning logic:

```python
def test_graph_partitioning():
    memory = SemanticMemory()
    
    # Test detection of need for partitioning
    vc_data = {
        "@context": [
            "https://www.w3.org/ns/credentials/v2",
            "https://schema.org/"
        ],
        "id": "urn:test:vc:1",
        "type": ["VerifiableCredential"],
        "credentialSubject": {
            "id": "urn:test:subject:1",
            "type": "Person",
            "name": "Test Subject"
        }
    }
    
    # Test detection
    needs_partition = memory._needs_graph_partition(vc_data)
    assert needs_partition, "Failed to detect need for partitioning"
    
    # Test partitioning
    graph_data = memory._create_graph_partition(vc_data)
    assert "@graph" in graph_data, "Graph structure not created"
    assert len(graph_data["@graph"]) == 2, f"Expected 2 entities, got {len(graph_data['@graph'])}"
    
    # Verify entities in graph
    entities = graph_data["@graph"]
    
    # Find credential and subject
    credential = next((e for e in entities if "@type" in e and "VerifiableCredential" in e["@type"]), None)
    subject = next((e for e in entities if "@id" == "urn:test:subject:1" or "@id" == "urn:test:subject:1"), None)
    
    assert credential is not None, "Credential not found in graph"
    assert subject is not None, "Subject not found in graph"
    
    # Check reference
    assert "credentialSubject" in credential
    assert "@id" in credential["credentialSubject"]
    assert credential["credentialSubject"]["@id"] == subject["@id"]
    
    # Check contexts
    assert "@context" in credential
    assert "@context" in subject
    assert credential["@context"] != subject["@context"]
    
    print("✓ Graph partitioning tests passed")
    return graph_data
```

Let's test the main add function:

```python
def test_add_function():
    memory = SemanticMemory()
    
    # Test adding a simple entity
    simple_entity = {
        "@context": "https://schema.org/",
        "id": "urn:test:simple:1",
        "type": "Person",
        "name": "Simple Entity"
    }
    
    result = memory.add(simple_entity)
    assert result is not None, "Add returned None"
    
    # Verify it was added to indices
    entity = memory.query_by_id("urn:test:simple:1")
    assert entity is not None, "Entity not found in indices"
    assert "http://schema.org/Person" in memory.indices["by_type"], "Type index not updated"
    
    # Test adding an entity that needs partitioning
    complex_entity = {
        "@context": [
            "https://schema.org/",
            "https://www.w3.org/ns/activitystreams"
        ],
        "id": "urn:test:complex:1",
        "type": ["Person", "Actor"],
        "name": "Complex Entity",
        "outbox": {
            "id": "urn:test:outbox:1",
            "type": "OrderedCollection",
            "totalItems": 10
        }
    }
    
    results = memory.add(complex_entity)
    assert isinstance(results, list) and len(results) > 0, "Complex add failed"
    
    # Verify both entities were added
    entity1 = memory.query_by_id("urn:test:complex:1")
    entity2 = memory.query_by_id("urn:test:outbox:1")
    assert entity1 is not None, "Main entity not found"
    assert entity2 is not None, "Nested entity not found"
    
    print("✓ Add function tests passed")
    return memory
```

Let's test the query and relationship traversal:

```python
def test_query_and_traversal():
    memory = SemanticMemory()
    
    # Set up a network of related entities
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:a",
        "type": "Person",
        "name": "Person A",
        "knows": [
            {"id": "urn:test:person:b"},
            {"id": "urn:test:person:c"}
        ]
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:b",
        "type": "Person",
        "name": "Person B"
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:c",
        "type": "Person",
        "name": "Person C",
        "knows": {"id": "urn:test:person:d"}
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:d",
        "type": "Person",
        "name": "Person D"
    })
    
    # Test query_by_id
    person_a = memory.query_by_id("urn:test:person:a")
    assert person_a is not None, "Failed to retrieve Person A"
    
    # Test query_by_type
    persons = memory.query_by_type("http://schema.org/Person")
    assert len(persons) == 4, f"Expected 4 persons, got {len(persons)}"
    
    # Test relationship traversal
    connected = memory.get_connected_entities("urn:test:person:a", max_depth=2)
    assert len(connected) >= 3, f"Expected at least 3 connected entities, got {len(connected)}"
    assert "urn:test:person:d" in connected, "Failed to traverse to depth 2"
    
    print("✓ Query and traversal tests passed")
    return connected
```

Let's test the search functionality:

```python
def test_search():
    memory = SemanticMemory()
    
    # Add some entities with searchable content
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:product:1",
        "type": "Product",
        "name": "Smartphone X200",
        "description": "Latest smartphone with advanced features"
    })
    
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:product:2",
        "type": "Product",
        "name": "Laptop Pro",
        "description": "Professional laptop for developers"
    })
    
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:organization:1",
        "type": "Organization",
        "name": "TechCorp",
        "description": "Manufacturer of smartphones and laptops"
    })
    
    # Test search by name
    results1 = memory.search("Smartphone")
    assert len(results1) >= 1, "Failed to find entity by name"
    assert any("urn:test:product:1" == e.get("@id") for e in results1), "Smartphone not found"
    
    # Test search by description keyword
    results2 = memory.search("developers")
    assert len(results2) >= 1, "Failed to find entity by description keyword"
    assert any("urn:test:product:2" == e.get("@id") for e in results2), "Laptop not found"
    
    # Test search that should match multiple entities
    results3 = memory.search("tech")
    assert len(results3) >= 2, "Failed to find multiple matching entities"
    
    print("✓ Search tests passed")
    return results3
```

These tests focus on the core functionality we've implemented and should give us confidence that each component works as expected. You can run them individually to check each function, or create a main test function that runs them all.

Would you like me to create any additional tests for specific functions?